# Train and test the real tiny iPhone pipeline — one Kaggle run

This notebook performs the complete remaining experiment without a second Kaggle run:

1. cache actual YOLO26 observations on 3DPW train and validation;
2. train compact FastViT pose/3D-joint initializer heads;
3. adapt only WHAM's input-facing layers while keeping its recurrent weights frozen;
4. select the checkpoint strictly on 3DPW validation; and
5. immediately run that locked checkpoint once on the exact 11-track 3DPW test population used by the saved original-WHAM result.

The evaluated tiny path is exactly `YOLO26 → FastViT → learned initializer → WHAM_I → WHAM_ImageStep`. HMR2 is not loaded or run. SMPL is used only after prediction to calculate MPJPE/PVE, never in tiny inference. The recurrent Python test uses the exported iPhone step's per-frame missing-feature behavior.

Attach these existing inputs before running all cells with a GPU and Internet:

- `3dpw-model` (raw images, sequence files, and `3dpw_test_vit.pth`);
- `3dpw-vit` (`3dpw_train_vit.pth` and `3dpw_val_vit.pth`);
- the saved phase-three notebook output containing the checkpoint whose SHA-256 starts with `f15875f3`; and
- your private dataset containing the licensed neutral/male/female SMPL files.

Expected T4 time is roughly 2–3 hours. Progress is printed during YOLO caching and every 25 training steps, so a healthy run will not appear frozen.


In [ ]:
# Configuration and exact input resolution. No generic WORK_DIR is used.
from pathlib import Path
import hashlib

KAGGLE_INPUT = Path('/kaggle/input')
THREEDPW_ROOT = Path('/kaggle/input/datasets/nguyntrunglong/3dpw-model')
PARSED_3DPW_ROOT = Path('/kaggle/input/datasets/nguyntrunglong/3dpw-vit')
SCRATCH_DIR = Path('/tmp/deployment_tiny_pipeline')
CACHE_DIR = SCRATCH_DIR / 'cache'
OUTPUT_DIR = Path('/kaggle/working/deployment_tiny_pipeline')

CLIP_LENGTH = 24
STRIDE = 12
MAX_CLIPS = 1200
TRAIN_BATCH_SIZE = 2
WORKERS = 4
YOLO_BATCH_SIZE = 32
FEATURE_BATCH_SIZE = 64
INITIALIZER_EPOCHS = 2
JOINT_EPOCHS = 4
LAST_STAGE_EPOCHS = 1
VAL_TRACKS = 8
VAL_FRAMES = 300
SMPL_BATCH_SIZE = 256  # change only this to 128 if final metric decode OOMs

EXPECTED_SOURCE_SHA256 = (
    'f15875f3fed12538312f59956b6c93e9cca2ab41a9e8d87edf593dd85f311ab1'
)

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def exactly_one(root, filename):
    matches = sorted(root.rglob(filename))
    if len(matches) != 1:
        raise FileNotFoundError(
            f'Expected exactly one {filename} below {root}; found {matches}'
        )
    return matches[0]

if not THREEDPW_ROOT.is_dir():
    raise FileNotFoundError(f'Missing 3dpw-model input: {THREEDPW_ROOT}')
if not PARSED_3DPW_ROOT.is_dir():
    raise FileNotFoundError(f'Missing 3dpw-vit input: {PARSED_3DPW_ROOT}')
TRAIN_PARSED = exactly_one(PARSED_3DPW_ROOT, '3dpw_train_vit.pth')
VAL_PARSED = exactly_one(PARSED_3DPW_ROOT, '3dpw_val_vit.pth')
TEST_PARSED = exactly_one(THREEDPW_ROOT, '3dpw_test_vit.pth')
source_candidates = sorted(KAGGLE_INPUT.rglob('fastvit_hmr2_best.pth'))
source_matches = [
    path for path in source_candidates
    if sha256_file(path) == EXPECTED_SOURCE_SHA256
]
if len(source_matches) != 1:
    found = [(str(path), sha256_file(path)) for path in source_candidates]
    raise FileNotFoundError(
        'Attach exactly one saved phase-three checkpoint with SHA-256 '
        f'{EXPECTED_SOURCE_SHA256}; found {found}'
    )
SOURCE_CHECKPOINT = source_matches[0]
print({
    'source_checkpoint': str(SOURCE_CHECKPOINT),
    'train_parsed': str(TRAIN_PARSED),
    'validation_parsed': str(VAL_PARSED),
    'locked_test_parsed': str(TEST_PARSED),
    'output': str(OUTPUT_DIR),
})


In [ ]:
# Kaggle supplies CUDA PyTorch; keep it. Resolver warnings about unrelated packages are harmless.
%pip install -q timm==1.0.22 ultralytics==8.4.146 einops==0.8.1 yacs==0.1.8 joblib==1.5.2 loguru==0.7.3 smplx==0.1.28 chumpy==0.70 opencv-python-headless==4.10.0.84 scikit-image==0.25.2 tqdm==4.67.1


In [ ]:
# Materialize the reviewed, checksum-verified programs embedded in this notebook.
import base64, gzip

SCRATCH_DIR.mkdir(parents=True, exist_ok=True)
embedded = {
    'train_deployment_tiny_pipeline.py': ('9486bee747b620d04b7cf4efbe0031826daeb7f9cd946635b002cce5f48e881a', 'H4sIABJso2oC/+V9a3PbRrLod/0KLLdOHcAGYZGyFJs33Fqf2HnUsZ1c29nUli4LBZGghDVJMACoR7z677cf88YApOxk99y6rt2ImEfPTE9PT09Pd8+f//RkV1dPLorNk3xzHWzvmqtyc3I0GAxe5ttVebfON80wu8mqPGiqrNgUm8tgWVZBc5UH83K9zeZNUPwEdfLgl+9fvAm2xTZfFZs8OTr6AEXqclfN8+DbrG7+VnwIbrI6WBR1U6xW+SLINotgUd5s6qbKs/Ww2W0g8aZoroJyuSzmRbYK5lW5rbHg0cf8blsWm6ZOguDDVVEH26q8rLJ1MF+VdV5Th6p8LbpIfX2yoCEE66JeZ838Kri4C3Y1ZB/9/cfXP47PgvKizqvrrCnKTU2jyq/z6k4PdL4qtnGwyrOKP8V4l0VVN8MlNJ4HW2j8CAdy8nL4D+xfcJVnizqmwWWLbNtgzXKzuiP8/GcdFJvtDmpnc8xYZXd5hUN6Uy7y1VGdr/I5dgf6CWM6efnTL8F1tioW1Mf/RYNs8roJ6u2qAMQDNgHbF3mVNTm0sCmbIJvP822D2K2hX0FWXe5wDqFbFUwKzOvR0bIq10GaLnfNrsrTNCjW27KCmhuoz8g4OpJp1eU2q2CI4nteX8uf/6jLjfwN6L2SvysYebmWX/XuAiZqnte1TGmKdc5dmJcrMdxa9uGbcrdp8orzYdTZfJXViAqRr5K4xBbahfHL3J+wG5TR3G0RvSL9xeZOjSgHfO4AXem6vChWeSoJNj1ZbG8AaYFIx3KtOjdX2Tpd5hkhDoYGlNzsaL6gokw3ay4BMtJ1uoQVcF00DEETPdbbXmV1fqLQWl7AiOTXZrfe3mGhzVbhr6zmV9ZHstkky92GMAlrBkp/y1j46YfXEgU/rLNLgXaqI9M3GyMxgbGs6gSRLPNfwu/XZbbIq5h+13nDFXYrWCaru6aYq8nBVXV09PLVT69//PubV28/pO+/+f7VmxfBNBjclatyfGZjgWg4vR4NxHJM33//Ynx6hsXzi5OLi2fjs2fPxs+yPFuejk7n+Xx8kj29WGbLxVfPT54/fZo9O1vmz7Pl86cX2VfPxidnx8/no6en43H2fHD0/sef333zKn3/4eeX1BMJOjwK4N9gOTp99tXp8mSZL0bj05NnJ6Px8vT589Ozi7P585P8+XyejbOLp6Psef5s8eyrfAG5J4vFM6gyGmUX0Ofo6OhokS+DOs8XKfGN5gpILsTvCSzyJgqGfwneAmOcUJO8LBLMpjIRpW62iT+D52OdbXbZKnXyiqXInu8WWVLUaXadFavsYpWHETemIVARA0yarVYClOj/3WZ+VZWb4rc8XOTXxTyfiKr85QwD2ub0BJZYHkxhsrCJgbddE7Zsb55tIGEO3akEt0nPFmGV3chmP+SbuqyoWTOBGwBGUxW3MI/mYksMSGlTplwIYSZVXl9l2zx8hB/083wyHM3iYPw0Ds4ixmcFS7Ta2CAZBkIzu8mpOBRiQoHeI3/YFA1sWDDSKoTlCOx8t8rFbADTfb8GvMfA3qp8+OY17B1FvlkAx8btI2hvH7BhEK+X+2ZTfsw3CTFvBIh4TFPYo5o0hclcLePgqlgs8k26KNZEfICh09HYmTv8V++20MMoUdUjnQWAEg0HQOgPuxAgsoIdZgrcI3mf/7qD8cPYQ1WIKHuTvMbN7W1ZrcPR8fhpFLfygTlmFWWaA2gX/O7V659DI9npMyFxakDUsHCig0cw03YNQnXdWWf0FdQ50XWgFGIruc2ui7xKd5sChIU1o55aT27y4vKqiYNLkBymx8nx6MDK3JG+6sUCsdvcuTTvJ1BeMfldHp5EEVD/FqqEgHsFjsQrsWuUKQhQC5NnWChNLoqsTubl9i4NZS+idlExBCoM1F8iSSkqhaGC7LgQREpk7Fvnu+0qPzeTY6vQzKBfXMOAC14RekWrAjyPUMKg1JBLS2YwHMEUI0Xq0QgS8jMnhRFBJIRZwVfoj2QnCpyiLwNDsnKr7uir2CQ2EEeuC6zL1c6TJImD89EYCiLjmsySdZ5tQiDU6RASP+b5Fn9/qHa5BiJYGnY6lr0ZCtDAvf6q5KgQNvPf8o2ozlztRy0WM+JvJ7hRbRZZVWV3gg/XH1uJKKynF2W79LzcLJGCcG9ZrsqsoVQSbCfBRVmueMsDYTNv0qLcyVK8Y1wAyPT27vYudGAT7ehP7us8RwEyvY3lr7s4qKFx3PsJaojCHGxrKO/TT+CX0IlbRt5VtlrivGGNJ8E4OTZ3CGgrq6ktzevOZYOAX6ys23UTboPHbglOmGnOtsBddQrtUFdPxpwj905EBKAnpAOIiQkYYg5IXrSwQ2AYMat82QC72CLX2wL7uC3WuzWDOp+MZxIE/hbbIvIkmM+mKdeiEjAyXWk80ZXgt5BNcGS1OMRMuf0QqsIhYBEa7VrAh6pzcQDcLxILk5pJYQFnfZBUZwCMMRoDDvdxLyA9FICkkWGBEoRgjfIJrIbb0OjsY6vFoVUaVnE+fCYnNF9vm7vUOIaGNGutBSiaNdI1AcIIkO3W4QnwEZd+jE0TEkESqMORWQwXX2oXEsB6YQFG9Me32Qq4TDtPDpIOkCmeAFJgfXBuCMWY8PcED2dc/qZYNFckvfD3FW2KRoJgEA4XCP5JIk7scJq0uUJGW64WgptwAUHCBpeJj/w4B0mXuwhIusUTaE3tBMA2VvkmNPMilIOPMUekKmWFrDVxObNn5p0BAKmabSSYAzI58O0rkN+YiqNkvt3Bf+mYKCBwb53KyD4PqVzD3pfzYm9zOpogIPezpzjHYn70d2/2ATzO7Dz/fcT9UYceNf9ttF7ksPqKzSLHwwFgHhc2FMeVqVFqbPh6s4HyI8HmcyDkiVsEUYc9UruQ7oYGV8IBcJVtaxt155Jhw9/YhMe7Dw1kw2OdaVh/Dn6q8mXOOjaQ1mvgMaS/IsUWqr6Uggp+zD/GWHBjUE6ipaU5HDigT6p7j3F5jgCxuvQhKCQ4XdhjapJNnGswYkyUYNE119BJ7Tow2261r6fe1Y2rzujO19YKVx1eFzUq/qDtroW3j80aUJLb2JuMQpE/R0pGdq4zQjvT4ao21s2jEP2UDCcl3c20xYdIp2Oi+RBmoLnY1G7gfAJy6HgGZPRwPjFrbyzG+lfNWPTSanzM8j4Is8XFyiph1PsL8OTkREgAqGjMF8Q7BHcxsSFpDqleAE1qEAmiCKF8ZdIREZfRYn0uKsxQPApBlqinx5qkhHDRUQGWlluhLb+i5BrKhh9LiBGLp1HSlKuibsxzPEu7JJRIWQc/ZFeGchQwukfA+/BkkYge2BxQCa5yyYYmIs+PZyTqGCkgOopuuTDuOmCMWjBO2jCMEVkLggFaSQLn7USXDKCvJOZZ44ndrrhFjINmWxnR/kL0Hp/aq3hkyk5y0y+rRbHJGtr7dN8BCQAi1OtwaO0ubVLxLK4IsYkINFbZqtxc4i2CnBJEGC1auVgjWW6eCUHeu9CtcXFnFck8Ua0MlWQgUuLuendWPcU5vBXFGa2Vd6CoccujAoYBg8wR+WFoTIQ6pp88jWKFCTj6ZzUCDQ0M7xXRDb4f+jjV18ioTMgesbxn1vmAO+uV2Dv3Gjz4x0ddW4yS4at8Ucwbc9+seXxrvLiakO6fq+CdTD0JkCmd44XMzBXe87pTfL9AMSeti99yQ+z3aaU5Z5VdYNt1U+2V/gPcDE67zgCUOxYnARzoOcI0eik0UFV5I4dmTPQMqp8zY6hhmA2pRvGaKwHpbQlYpzstwaFR8KNSKPpVGdBueBzTiYIQB7SmkWBo5ko4MANhrmTzdJ+T0H+N5hldDfCrskoLzNxfHjuETWN/qAvn3L2J6OZjo0MzW1VIykSGXG7FCCK8f+Jr30mLV1aXF6hcoVxceSAzNuHg3Xf/NYhaZdWIk2wLwBchVCZtZBi1CzuDllW8uwDCAeLAS4kQRJIY5RJAO4/jXV5n6+0Khbb/+uH1D29fvXgX9bB8lrOQa9M6SMQ6sRt2OmdzsWJ9Wf82xX5YyUjKqAk+HsUOLCT/qXFJmVDLKWek8r5X3OU4unTUNk299zd2OZiXi7LOp44QqsfNJMFr2t6zaGPAFY3ioT4o1zYWrdOcPlxbZVAWIeINzaIt6jTbME5RkUXgkphiMWPyNIZE/1uxDRWxyQJ1bI/RUZIjI+gksg6FR4sOuSfevEeyQwnSqb8M98yf5+OE/pIGM4wPlm1g+kLG/5MnBnuA2RhFwX8E41OlEKHZAWSRikRzORubpAlGxoCKwfIm4W+cN/giPokQrCrbCmX1VofRKCFZwBmm9uP8kzeVLunmcBzKBxPeVOLucmT3UWNBObSewk3ZZCtRVvD37sI07BRtOaAG44B1i7olwG8PAJjLXUN9Cz3bD8qyvD2hSHhm6fDMf/dtMmgXXK529dXUFh40jQB96In/k3/iq6yA9f0OOgddfVVVZRUuByhGBDQTWOcSthdpvDMJPimQ98F1zZ8M9H5gyWB6kge3A5I1YNzzj+E50tatS1czY3QDPMK3qmBiby15vG/VlBn9tdVa5fpK0iMIWkp0YfTKe0xMHoDetTXrUQkPtEzoAWeoXg7u370QKqUaK4XiKc1kqMU6EFy0IPZig0cb0nZJgYatZpIPCOMDZkAB2l8BWtlM2BaI5DktiU68AqqQQEksMjcu67q1KVNWjXDnzgfXgN1ZpGU66hyOXfRSbz/IL9JiUbs3uAqkzdh4XE32EUdyU6sGJZyBxERSbABDucEYzSMEsMezp5agUq6uSTIVLdR0cQ8bBGNe406ClwViPQR7U4XuxUp0xF3U6las2nS4fbEUciZycIARzXD7R9s1W6Xbyyhe7kBIw4MbdyCr6+JyQ4ZuLM/CDrwgavwE/7l3xEu3+SmlHNmKwnNWhSI4+qWHmsO0kcmdYD9yRFKKUeokAWwPz3tbQvoNEy+w++1W2lgShxMwgOcprS+Shc3xzqn5liRvdkzid6bscIDJpmoJtg92XACr82KK+9YmH/H2rE9Gi7NGxaERpM20dZQkEeqBZ8JtVV7DaZ7umt1OGsYGxvHUf+AzmQUe59q8ilEh2ZLJfdQmqHGIZlrLwrbQ2mZ3wBfpsEjDwI9QV4mRGFKpeZgO5tvdIA7YVKRO0aSU5XNLLBMwE2DK4UDjYkDyl/4mmyPeQKn8udgvZkpOay8YzWYmHlFsOXiXk3EtHh6KZQHLjzZzbUyLA4NdXA/wHoaj5Qh7iUoDBmw49ohtH/O7iRzsOXy09XK4GiADF4NfFgThwC8AsRTQkaf2+q58vZt3lGBMd2Qam+0+cfy+pVfRzF2YKqd4a6V2EPwYiCOSZfo87dHv2KszPrJIoqU4om7oVGcBG4o5Y8XSuNH6lrnRwFTSGQsIeDqw92T9cVFUIX/ULIAG+S2s5rT8aNARr6g6u87DT+Y6mBiLIA4ePTIHfB8b7Vn8VZChWViwUW2dnYp9r1+GabE/FPBTyTlhIRkyC3wJLnSR1TS7CEnYryQV2c+Gg3SApwJUxlsWLZacMhPqsR1f5ggr65DASkRD7xdCA60Vu7TzWZu+2Auxqr0XGsAk96P2zjFjhmxlRFzHkDSCJwQnQuaIs8q11faJW5bul3Et4e+r6u8h/aQj7L6eWL3R8xT8Bc62koHqzkSQbkymuXoKuc5gg1kVmxrWplL7mQCGwcgkCEOCdsQ5GwX6i4Qa2WDEJCE/SSyQfdmLZI+ogpKK45IgNj++n0aJSwoyjoCiW7DED2P1/B4CiFiBE718fk+ZA+eFD/+/jyBCnbRzZpP/8ZLDQ/Z+g04+c/cXwglT2WCm6WIx8eJPq7ZZ67dgpR+zBYsNqFMK6USmIwO7wDLzMmWNFPBbi5Oem1YKhx3v2ge4FhBlwI+K5GJDCh996GK2gGkSFJKuBoP51lRqSkVWZc+Y25T6jo1qkb3Nu0NTR0deWMbRXWu19WlSotM4R55PjH48DoxbVcOk5iESSpeU4pFU8J++cdorqfikFZp3FlkAHZNPgs7un9CJTR7L7yef5NDvB65diFK6To/b98Gayr065rY0PCBKGEzErtfOlx1BxaKcjnYpwhQUITGDL3NaZ8qZp56ay4Exr+1ytrQVH/mVjg+R3yRjmAh0dUtwnO9x3PhmVWyFY1Mo/hpMxbJCnwkeYblfWI4Fekx929TBZ+We61R3hUz8p1hdGH0ZU9apGtsX31lWQBJOIvICrFK76YJyta7aMq9kRAgPKCHIe5xQ0CqeMQQrm3/YmUKsmAo8Of4egifZp1LKcpiGJbBbBQ10oN2//rIQJhHPOgMcEo5r5rlBpX6mQuhUukC93fA4nJOzsMhJ+SqFz922ag9Ye2h29bHJ6buvk23Aj3H74FmOPOq18kYhWrbbumRz+tA+bcsrHBPjSqlwTvrgVh008ivLVUil4AATkVCtjPSLBXs6sBGWh/D8N184bZJthta00GCMy2PhefeO/rBTXFJf7ZZLELYITGtTpdT2nsrJTEawt6mSM7M+CdpEU0d7tJsoZy9c72tuY4vesAuyDOVbZZStQDqE09wgalO46pTpNwbTKNzGaGkCQbesk5EQNZDIrA3iYdHka+V4RrjVDo8drHNiMz5rRqSbDGPPMMgzqNNgCucGhFknwbYWuUW1lmmE1dH2yjauNFp3wJa5LOqSXUU3TnqLMVnrQgykTcy2JQN3Utnmj0/PxH/4hMjZyuTryHMf3BSbnX3B3uMq4+mzUnyZ3e41B9HM2qjRax+CbZAthbZuqH/dAfmnmNNh1OezJomljbEYoptwpxJwwNEeTZs9FWbvWM4FUb8h8gmxm5FYL7y6b1LhW+aYLnqPCq3LH2OnlFJ/zLfhdg+7rNJUvrRmM66S0WftxGcwITp8gLth95DQrDWF7WiVGx65gkqrci3GK9ET+bqhnOgcX9+yScUszst5eQha2xYavXjmhk9eKlzbN9x9uDZM5HOUOKvPvPSz+iMA1O3+9PblIgdp9vdonwAd1jh7K05GxzN3S7EPLQOhJJKKE74z50TH2oDu8FuU42FQt1LQkFZiDhxxt38IKCraD03erx8CzhKDOuAJBTqt8Qk7bnoLMHH6GuacDrg8h55alNGqxMSbKsrzVBRljKryXp+uQC6R5EihT4EJhHXkDv2IJzZJCif379+8G7/nAlKbp1zqJ35Pey63yZubskJ3VOl3H0tbzMtcWIfuUbjhPk+aA7otEt2MzR7EshlDghY3zdkaJDFUOEsbQJlUu67VKgfY8a+7AhBLHthpaGjjRFdgFAhRNJpwigUQ3RXqYgOjhKNwyAVixMFL2IPKXeOeNahAsiVT1+Mjb/+N8XaMonMErMOjcpdVuVMnJ49+7sivxxgQbDKsQpu/rr64pLqqoMY4Hz5FhNCUk1GjUX3AloSn+fDUpFVLuc8VcfKZ+w9igAx0mVLGoG/SBbUkNUg60GKKARw+hwZsLSiFDoLqAo0wrT/JioRERRjAqdJ8fZEvFsXm0pZcNRk5oRkUUdE+nm/mQLdVQlBSikgUH1J+k+8qGC9i2l8e/V5hbMA5euEt8h54DvZ8S47IuhPfD8C5i3sp8KnquijTeJLfNi3t3PnRYUaHmt7POy36tjzgg8iMdPj2AL1wZx03z3IhndoLiZeDWkJn7dr37aRPenASmbFoYdTbArR/5sBzdcNWXWONGr6gIEtclBvUAkr0dC2jzgUj0S1hJdRIfX6CPkJu3hK1v+hdcf3vIFfPcPdTrihgTJQHTKyp4qt7S53KUFQIpIb07vX/iF1eqxrlPGH7YeS2lpBwItIlSzKLPmRX8K5Os4HD+O6hVQxW2VlFc999UD0M2Kqyf8V1r5ZW2z2Lxyr752C+e/n2bfAO/o9lMYwMdDxfyFB/KGDnwcVuia7S+XW+CW7QDbqAI6O46QxQb8bxTRIl8FSAlLyuyULA2Bc7EK1Kx/uQ55Q0DZ1lTlIB7S5YOjPEsHYZiQexvBhjKQWx+bw1JtRdVoQdMwZLO6O1rNqxuJYrcpwSyhEzuI6hoPJQkCmaTR4QmIjGT3eyNLQQ25deq54N5SGgJRbX2VY3YJAoRzFSxyFqmpyg6X8n48nwZGxc527Kak0DXBjQTNYQGg0aKgMxQKP6I1VdnOPqZoHaTTsR1eQ+RHjXah8aTedfVkz/De2JWC1NgOTtHMOw4yo9EqQgojJh6BQKs2RuHFzzERNestvUv+7y/DckG/RkFoGaSKcpyR8lXOCPdBVA1k0uIYtgRAdEj8tvt9lmQbOCdbBBhihbJGq5KkAGkEWTDYZC+zoQ5fBLo8iAp8pbQzJHLpqCscuyEfuYB090bf6RZrUoLYok81W23qZ4X0+u2owZpYlblXUdigtyjF/i4Iep5CAMaSDpQ0Lt6WoynEBkGic+BBRXscFUOdBocZ1rMwCjg3+1W0HmuakpWheyouFYuljDWZnUsKGEliyK7BIjZmIsrREF04If4ymSImId/qJlBeBbutrTLOg9Y4iu3Cp4iAig8jG/UWY5rNXSvr+OjQt3g1Vm6ISP3uZ26ghDHcQ91Y6xQKsaADvurTbCAq1qOBijmnHSpmBjI9MwU2DzODkFghZGSLiRXybX7OuITCxEbMSitsMGsEbWZJtxiKBiMUHK3xhtOTDoSNgr/N36tq42O+B0yb0ftA/SLT9r1jqzsbNu5kFx7byXZATy41YR0RxjJhjdsFT5eLEBk3k6gr+3tCONZpGNcRF+l+M1CtkFVpFIDqnWaDID4oatLoyYnYrEyGQKwIS5BzFdJ8HZ14Boy02hajSW49HhwVAopd6oj1TfdoYCoGojVvMm+hTzZHX0kCPe6W8RYy+F/7V7K2S30DB3ld2L9dQKtBrELzm6VotZ7bdm6FZsibTCRzJaYOypLhDsyeGBWE75Xvz1rheB099ldXTsIIPB4A0F28Zw1yndBr5v8u1/1hjKSATNRu/rxS5bPbm4wwvt4CK/yq4LjL99m82b1Z2OdGrRi7EeWsThEL1Vz3MiAsa3G4WeDDqGjUJdPzoc4LgT4PjhAKneiVnvSJsbSmsLlqJAjNHFhCjTLYeYEB4FRk0082C0RvATBQ2MqKdLRxw2CgtgCEmejWIDxy+yBUQNAJxZkDA1Of4rebgULvp5+cftg/i4ymTr4XbmPFvjPec1WhF7FkivPQSQ+d9otyRZH3U90Hf0YSsx9hzGbsXkWwyuDT/sBQUNr1a1Xin72LzN3393xq5BsdGn/gbMlHMOYCthgoj2Dxr3XZsFq5WtsB4bODaY8JdsJbFNLAduIjoBOpRXmZUCTeDbANNunYIcqneTidrbi+HGLG8F7R7q/I/bk4WV7+4jA0SkVQIT5I5qeA2LybKKykSjHGHEKmTsT8KdHlFkFRFYszyfEWl2IU7TnsKvf3z/Pv3l1Q/fff/hPaCXsTJQiEP1pLSgHSiagdQzO1Xdm46Vwe2ATpuQdJyMZRKLoXYaRasXbT1NrFTR1jM7VbU1OrYzJNooFI2+Ll1vd0C5eIr79+hQL9jBvoNXdSmEPMJtl+Hs5MijeZAKLXXdSt04l6YBM5Ugrs/jwNBAiCGLpSq/lMmIgRHWKURHppWiDVgz+RwVHRg+zzpZ222JuuZ9/SxyzgoK0LdJvQYSuUpXI4ZliJpml+V5N7YjrJw7t/4zT7m8ydCG3LjbhLW0m7PnxwZ4vuVQJ+D9hoxKNuFe9lPUs7aGiQKJtbRRrtqAIPNMPxQK1WIPBIXA0AI9VP2PhGFYGBnhsEfmWV+BYEHm20Qk18W6WGUV8LPQ1okp3AadiNFFuD0Dsd27h9xbHV6Jygz7lGss0RYd3A5mrTS2UontUbRKydXTQXm4Bxx7cvUOI9ViwL/868Mdum+FWMc1zSQPWCguIg9dKvJc+6UrRjPuB3WWauzrq7Cs2tfV09gK3lezM6ghH5i7oakv1UxNnJrNuB7mZumvJNRCs67Kap9rVTcm11NZ7rtmtdbi9x/3LelBbNUmGHPt74VgbuyavoXVuVtOYEqXk9jpKK+Q41J7R3kpG7j0JiUhXuFNthKBkky56HwDaJuxWn3HkWIwJTbcgYlqEjTdrsPI0cU16CPHJaTSvbzO2R0qPERC8LpBHnAAkpIuefNTX/EwK9ytAliK6QWInh+LzSU7CUqnfmNg1DM5Li1SMatHnx+QckDEyLfsksKjTTFBuLYEN1m13m2NJDdyPt2pwInra7uoa+GIkZqoJQ6DxaGbzBoUvQkNIE6UZo5u/XCbo3pDqwEJwuixUyIO7KM9MjUK8fwctbKsm6Vz/GN6oQt3wJB+bAvIwdsE2QUOARtFZmSeNN+Wcv7/tfLoih6cmhiPT3W6ZHGFctvArk7NcTYlJD/KZCHCCnJwSq0qRShV8jpbXyyy1+9iHQ5dFc/W2+S7Klu8p9R9/r+mwMyDuuR3ooRvlLM6iN7EstAGFb2miJZATFRC28IAVzIdMo4xwsEn5AT0pfgCrhuTfdwzd6HIh7S1HB8YzFLZd1p2fcqAxDWuOci+8iHmlbb/stdqxrDzY1+pfBuTWT4xDttJayUeOmu7BHPhqckXFQzJ/QzXVUl3/CAO3TjjnKLxNYgXjsOzcT2d7ZpyDosq9LifpuToYQRvdHxUXT+QkWOqldNELfaFf3RuxpGWYjqiQsf5bOU7r1oOGf00K3BGpNsRXpCXXUJ/QmwmSqQJSNgqtdvQjzRUWI+ct8iAu/DDduQPRKZUG3qCSZGvEQHbbB9oxQdVNrzFgAZWhyQXoZputEyKkMhOfs5J1wAhluDjqVHJGA6u8XNe4DMsxBIjfquA7qim9dT1yAR6VuX+6RpcUHMsW6jGeJve05rcMv9Dsz0VEpIyRJQhXnHegEK/Z2xHYQPBRBf3lcu3VAyYRH8pGQBSDKCntODGztw9kVP9/0I4Rymm8U7C9KMG4KMsHqwrlRlOESIUbtgKyAd7IYiT0tfMDhUdy4so43mpjluoQxz8pGt7HNDzHEUtLkVEMDsR80d3RR5kJq5Xp6w5OfpD/PdavnsPCrv8UJe6Xnc6jpJvfY7sT9MMyokZ+1Anupa5gOUktOcKyogsA0JBVcz/LYrVvigAHJEgtQIBON4SDw6K7siUCOQgU1gj2WMJS2oqOosqO2odYUCvJ6036C3GR1umwcUB5YHrMZL2RXxRDtEy9omUY/kA5SRaMTvtubAVage6CftjuhPEcxFYY9YV3J1niNcpHMracde9wHw+qrYHvsVlrZ3dgsTO0xPsgcc5QJQ13H8PKS5d3jrLRv0nfatsuWu2ZDIrbRYlD/AWEj8eeVzWOvtGvfFElSc9ht3t9sW6TS6Sw3E3bEM223ZSWjNwPROyIYveqqLtgWA0P6uWxqn5Ak39sQcGK5H3g3nohYs0RjIAADpsvSVJOYLhYGxhV/et3D9vWzeVpkenYbDkFLD8F0Wn2qXERbZyclQWV8b9jDTpGfXZW/ngCvViW63u1cOaxYSBsWvJN9C34arHLFe06HcUy+WoI/SAdOWXOuQxduqjetdxzlpwXnMP17SaFt1035JEBW0XKdpoePSICcoXOwmP5iK+/76gXOJ2QnSDwyjNZsq12fZbs1+sojbPJx4U28EmVE+87u+cG1n2845nPu+R7lD0rpxm+G4wvj6pnO5DD2uUg00M7NKjX9JCGLFveHebT4G1bkg/tysmgXd3ZDLq64h5ebJ3guUVy+fP8CjommPH090j2bTiPsipYG1/ey7kYPVVJw1AdwcIPxjbkVmZPuS+w1++aVNFdFLU13lV3pMXOe/NGGLXYw6HYq9s8ZyarcowahjyjYxEbLyILCTAQ6Ohyfh0AzVxKv6ZL3yZetHBwyzbpfV4xUsNvMM7wxWRi3xB1vRiWeSXqr7AeHc9CnzRVY9vcXtqX5SLu97ao0lfdWN7TzsGoInqQDgdAzKuH/cNiumyWtd6GvD9hV/h3OohWQko6o88J1+Tt58GMxaadXxqlzTWW2QceLCX5uvD1Eunqm8NRqrb4kaCX/RkOMRFOR/4Eb+kJkUHhTmVozvSZWfGb90hbVM7qArDv+bVrQy4R8HD5cI0LzOt91HItM3IbC2eFvcw77w4yX70pGv9GKiIHLu3jtJd5NW1VmQd70o5dIUUGEunt3LPsoAinZ221oL+MBEnZwvt7eRv+0UO2ILQXwzNKPEVK/ZWNrQan3Wjq14apQ3Oc4UrnKKNhqPWhe5VPv/I7+aJoLBCncMaL6+C5V+r6qHLUkNB417/aYWUv7drwBI+b9oRLNin1NFRgAlpjIvIzJJ3FF0hN6QlnkY9htK0iUCUcQ83htCP9x5rNPV8+eqn1z/+/c2rtx/S9998/+rNi+46JsVzvBio/2lwVSygMWh4rQ9DpKHSGfeHAe0bklGuZ1hsiNEDRt5YdoMgooCK9LcHgd3XE2YxTULyfSj66K4gaQrPyOKnUzibz/Ntk6PxsOfBYZGNAWZ43FBsQJdmsD0By9ZB+1P1SqpMNhSw2U0G+N5cpk1eNwP3KOkE2ie6Fav+pgI2kF4VNZqKh0YkcPMhRkcf6MQAECp6O3ofN3akdOkUZptU6YObAV5V3qxgiNMB/Ca7d9RKDXbNcvhswHr2psozww+U+omy/ry+Tl5Cf36hhJDLxcGyyFcLvCKppxTbBnuDevTIgZDQH3QQzs0Q9GYmRefiOFwy+MJqSYgN3ZFrIoeeeTlaqFfV9OxpZOp8bS0VRsrchGPy7NYuvawK2mdzixErqybgAx8bPsKhiGGxI5RZTKhb3IKjr9SJ+EF+qcqyHp/kaNgLoctPNPiryDL7I94sVtWHAiP5XR6e4GuwFzWaoILIEhl3ol9TmJM2HGFySYZj5yP0+RzN4BA5M2xYxwfAdC5IB++BBoZIAyI+6CQQ739YREA4xWeZBKrgJ8Zb3ear66IWISCB0geOj1m9hYWdsrIlhCMprDv4L72wlLxFmsbHDDr3KOYOKtbvP8qLVXHB8eoRFoc6SPm9JmWj3VMec83S4nkk5kL0tijeiMpAiu9e/e+ff3j36mX64d2LH96m//3q7+/pRecmNPsV2aCu2cSOAbUg/e3Fa4bzT9iuWLlwHwmguutR5LwBJTqIdjC6md7XFpxo42/Ew1TiaSuQoOoJY3f6yWrjPjb2Bp0HafcDN5SOfv/CWU70doCM744FxGxdVTlsfsDe9TNHKhi8gCMQJiCoXEY4gdJBrhCmXR8Q5GnJNGCXkaFFO/jSgXqciTOtyY3tDkb7XnWy63a97aS2OyYWzyswmhZiKxq+omIREbrjJKarkFxQGW+VWFK/OTZR0h6vfXZTo607HqUcOLgUxURn3fcFeciqpHyeJvU+maCfZaF4VgYGTQ+kZVHVjZ5RGrYYGIeSPZ45zwjyc6uIV94C29xJisuQiHuhKvBCPJb7E+WE5n0e7AQUYRSjxU9bFV7mywwfif0+X22/lYVNZwACmGSLhX6QdzAcsvJvuCgqkCxIAy+kGTZYWxhXXx0Q6KLySwDQuhrCuhoSxXwmFEliLSD9beOkDpmBfW7LQC1fCIGPSUN9ovxcQHg+GFYg2n8RgC/vBz5dMT4biqBHn01Zq2I75FjZEgRZRy+Y1KdK6OvAKoWX99YcjXtrgpxDjdcdlY+P+6tzXPgh8aMhaZUEHH6WQEE6Tr467YVE1+1DvG73I6B/LuEgmFf+MTzdO3372j7pb1xs3fugnPV3xJAVh3RgrT8DDyTh9lXv7wNGLBrSMbgPxmgvixBbkq/ys72VhQbTOxF7qHFVXg7J2tGPutM9XFVzNbve8fjs+PlotKe2OAQAiEw4M+HBGUWvXT7YM/Uk5Q9JWdBTXZ3SCYq58apgTRi4zD6IYgF6rcMoLYRjIQCKE6z9QIg41PoO7ZKrWbdrreNEbGfpk4OTwRtCqhmxk096oM5c5r6p4L6+qrhFBE+CARxknpAJe/0E05PtneVnZrw8639+R41a6DTsx8H2PTr7LZR6WzbflrvNQjwzIY8UGjAd8ibBIHgcYKzDBFezeoE2MjSu6KDUehXqKhufnnF3/IiN9Atrh9S3Uasjyeke/GkavP/x53ffvErff/j5Jekdv38BQB5ysHp1uyVBlA8ULCCJNoJPXuhwwloiGoNPuivts9VSDxS6ie+hwVgO6B4/gj4+Y513vVubT6BLkPJFYzjnrwuyjNpdiNAMCdVLWdbUgz0fXBYUwLLKr1mCwo/vX72gMPfzm8XUpljgQ/ltQ3IDE2mCW/xWr13R9J+cWaQoFd/8+ObNDx/2DPPnTS5Rj5UEQHwjjn7IIQrehA+dTH3qiMhQiRgm4vhqVL5cIemQKlkCuecnSza4kbUfoJM8SbZDGnkfCxJrUBgM7RYZrkR1cRz2v3M/eBF88/PLF8F3P/2MTzPL9TcQAbRxH2Cb+eYKFp5YTLlUebA5jNLN8WfIfhyR4rgJEwC+Znnom6lUjd/PekCtP1rD8/+tkuKaYrGSNuELVQxepSE61iH3F9wGVYE495iEAVZBBthlK8O2y3raKUkS6/nGI/v+nJ/hC5jtoRrcy9BN+hHPtukHqi3/ZnXDNDJ1H/j+Ld5IGNcQOhQgGfRygwPzEpQoK+W9ZjDZswNZtGtAkWyYYcgvO7KJeAU6RSKpr8rVQsb9MP075CuAHHDEcvUVTZMTAn84j5vtfcG9vaZRDGFYYiqsOd3i0c98qEnTp/NuXXzkeeix97loNf8HPQftpwlTWkLy/mKaMdbUFZAkyJupMXFfSjwGI3sI6YjVzVd8cq2bVKNe6RxM9FIXphIGASkMKVLpfW3XTywIZD+pGIzISmt1/vehFu+gfWTkIxKTiJBH6eaNdydpQ8/X2+aO64fyCTR+INK6TzMfkHzg4jHXuJPofymTJ0i/auaeZEgJ4ySq1+jc9NaDei60XB6VIsMrO0WHSOfBSbzusRefnjZJpL6JHIjX9qCM7Xo22EDHpEpFABCf5nIs0LlnXVZ3bQBbKAqiCt3je+HgK37mcpGSn5GtZTh76OckWuY4nmWGYcwGOPqxgSQkEOW7HgqqQWdsG5B5orLfcu44mD7oXWflhyvgyJ2iLUZlwNOEZ0VXw6Y7wCE3yy0fAmEs4W2cT9hoOu6eQbwHcKsz0rDBx/zTa6QCMpcYWUZoDzJdGbTQofm3PgC2y0t862IyxQ39wQeewUQcgVrZnob7tx6nmrv/ADfv24FMqedLJSW9CX7ZtonKIDqNpbvaZ65y8L2VKsc6Zy4mVqi3XLe9r8Ome4x/BwbPlnzIy8YHQo8+8TN08h6W8jMUMoM4uH3HcyivLMBm035nZmA/HTRwfWs6H6nYU9J4RWLQ8nSR4U07YHheojBKWteE9KxDKljFfFdVxOoVZpztgFSrE3dfY84v7Iv2uRdeIAXa5rcwoKU4clNut41f8E9SiEJN/KNrkJJb+SRyExy+P+BIVDqd1IBT92xPEn2xuUu3xTZHq6UUC5N8ps0Qa0tRGlovMMRS4aFN51jzbpCvfv6EyopYRu1SxnsDoqhOsctH2mbS8shkCzjOqPnZI3LkNl2uRebX09YzvK6jNL9SQyEr2i/SHR5axQrpofZpDh7zYpGtfwm5IbkVIxlndxhibOQ6PHAUnynZOBvhA/BRABpVbL7vzKF+hFOoGWygFRboyZNghNHbI08wCqfD3mg3jo+KHTZHP2mOhUUEFWp9avSB4yhNuc+TVgQmT5QQBYardjmcuXE31HjscDxCARarOCe2GykSV6rdcwUx2vTD5Ph4Gozayen+YDiWAM9yViuOki9OSvvlX9fm2PN4WjtDBK9ppXfMpkUjvqzMm+4esfRgvJE1mAnIECDxUfcbu5p32souy3v/D0Sf7wzbOkd2DFAKNL4zTh/iIvc96ZZ3LG1ePVbEh0Q3efTo01LIdGiWfz+YGGHibEt8SbrK/N4LzRtmZEmintWAv1yrUT3ZsllvReD8UA/vGwxPhkMeo9sfR8Va4f5oKnYUFSEySH+01svfrdsIKNF772AMUqPjvOWPMwu+NuSQ9stxlozSC8hftWMR+gvL3Zv+dnRFyDLtYE9a11Jn17k/oE+X04fvH5+wuoPpdHKLg7jGXu5h7RV9veiNP9Rl02/+8xvy94T2sQTIeM9L5z3hl3iRvxdme2gkT2DNGwI9YZPgk2rzvn1T0N0B2+TfI+wah3d1RJPLcV5fQ1viS6jsYEW5pF3ULIxj3KKy0cjRl+n9t3cf5Aa/rcrFbo7IKM39yzDpkmYTGML/D9RQ6PWIDzmqD7eE3Cj0R6tEywFQsxS3rOWO4qS4ZQ2UTAyEU9iozqIHaAwUKPOw3ev2Mtjkt03KFi54EVDtNmlzhdHs5hhElu+SEGqJdwD4GNcCo/YBte1AniXnfXUovTdmt+d85iNZrpTgFjFwobCnR4r379YeQiWMbQStNP7Pxuek4rvw6wqf5hE49pLT4WTVQzJ+OUI7Dh4oPjiA/yAZwkeVPCHnXWTrOI87UOUset/bcsOxoXkV9D1NccGkKcVtTFM0tkpT8bQfW14d/V9Ln9+P0MMAAA=='),
    'evaluate_deployment_tiny_pipeline_3dpw.py': ('40fb6625e9214569726b7db53059e3d296c7a6b3bd6f74b8486199afdd3ce461', 'H4sIABJso2oC/7Uba3PjtvG7fgXCfiiVk2g9bPmRUafu2ZlLe7678TnNdFQNByYhiTFfJUjZquv/3l08SPAh2bm0N7mcBCwWi8W+sfrDd0cFz47ug/iIxVuS7vJNEk97lmV9TLwH5pPp1ZdfSM54TlZJRvINI3lGgxhmvCRKqZeT4AssYSQNUhbChNPr3W0CTtiWhgXNYVGYUJ+LpTAU+DQPknjIWci8HND4LA2TXcTinHgb5j2kSQAfaeyTrIjFsh4P4nXICIyz2IclKc03F+Qfnz9+nszI8E/kR8rzvwd3+DFkNKsRFwd5ALv+m2U4/cuHyxun9ykhH25uJyRKfBYeZYz6SSG3jBPy9ebLx2qZIBZo9ACU0IyRgjPfIeRj4LEYPvYEOOWc5VzM42Fh/yQOd4Suctg2zZgfeAJPnhCPhl4R0pwJhqQ0BYiI5VngcQfZ3uutsiQirrsq8iJjrkuCKE0yJC9OckEO7/X0WLZOacaZ/u7xrf74K09i/ZkX92mWeIzzcmRXfsyDiMk9vSTEO8Ed9KY+W9EizJF+CYOsD4N7Pf8FvsqJfJfCLenxy3hXEqkEgcGJwtDVYuKCGPksWa2AeUTMIFxrTZTcByGrVk399BFXqPHONY8bGrkrRgX/4Ow8D/JC8B+3UuPmyl+TeziT/hYXUbpD0DgteZRknj4nSr9bCa2bB/GupE+f3+4R+HN1/eXj53/cXH+6c7++/3B9czkQw18//3z7/tr9evfzlZj6cDk5mckpKdO1oatyq58qWZZTBhVekrFBry9phBvLaLjLQaY0QYi41/v75e1Pl5/uyJxYguxdEiaw3Qr0ZxvkrlIeF6XfpT5NQT8FN0Eq/0Cun1ChpC4OQWxBwEiapCjMyNuIUQ6M9cn9Tog2p1v4koGaU9CTIWoearTTu73+8fr2+hNw4O728v3fvgI1z+I4lp88xjn8dR9p+PCXLPDXzB2N3ZF1Qcbj0zN5aGsFG7rrIshppmdPj08HHSh+TjcBCNZoJICmZ8dNoCLlOQ0yrkHOJi0Q/FAHmk3amzH2EO5uaPbAcg02Hh1PFBxYFz9JAMOKxR7wT5N9Xm4nzgQG6+EvdF1uNJ6cTpo71UkZj8/OW7QEMXz+uknSVGylIM9GJ01IkBuWIWAJND2Bo70YN3R7/eXzrRZRuCgp15bHRpPRyej0dHV8fHY6ORt7o2NvPKLsZHVOz9j5bMom3unMPwaGno/vJ1PvdLw6H03oyer0ZHW8skBSq01uru9uf3pvyEFK3Sj9NQXtj/DeJs5sdj49HZ1ORrDbeKrOYYCcnDgnpydno/PxeHo6mc3GCiTdKoDZiTObzo5nQO7ZbFyynXoeC10wQoEHiuVOR6uUI7Qzmcwm49F0dDo7n5yOJsiUXg+MIZjSKKIZqKCN1obxCxIGPF/EqRP7NMvobtlHP4MGc8HzbEBW4BBy8h90XssLsWuwAjeTE7VeDOEfsCuckdsiRoN8nWVJZls3wjEQILOIUM/AmaJnjdJ8Z/XFyoyBNYtrVs2paASyvCT2wCjG8FeR3O/rsygXDPoO9DJuh/SehXCkinqw4/JA4pTVGe53bkwjZoKWEEu4SMNv2DghicUIAvZiTwM47aPAAN8JA4vLMiRQErCwtoFvLfsVb9R2uJGtV/aXDk1TCAhsgVLuEAUcDRRQsBDYcUe9TcvuwEWELLY1cvwfHPa7ORkv9UUpfAdvaWW9T4rQF5daxMG/CgauP2M8CbcM/l2xDLRehE3eAzD3WeF8qd/gokbGYrQ8SPtSXSFGG6YzEhENl1paRVOuDJjQWUvBFx4yY2nSGqwWmVM+24KAXEg/6Mhv4G1QMPIiDdlCXq8phComwzjra174QNyr7kxij2PnJvGLkKmta7I46CkBNELFuVqIvLAbhx6QiKZumHjCR80tLy2sAXlkwXqTcxeDtPmPNOSsry+8Wu+sWW5bBm85zEXUEhLS8uwHJaScE1yyrp/SVugrkZPnFuIXMCFJAaGp1cDy3CD1jy1S/9h/qRYp/WA5+PWcAtMOnFRDwVlBBp9fSm1Q43IBT4rMY4bEuHxDwU1IDnUGOW/nknXVmRb4gdSzJAvWAdo0IoMyDKWDGAP/dAPhxjDfZIzJ+L7JgtY5MLlx8auLsT1QDyYW9xCC8RsovsPIEI0PxNpbMLio9H7CJDIxBnSCL6gyqkcIMgFfUsAJ/SadXGkNceGyaqollF5NtwVeaqfEAcZ/FawPX7aRIbkSvn7tRgI179ZeexP4QIvrB9E8QJIEFrlRNQV6dzKe9Pv9Jlp9HgzdhcMoGVFRvdhHcbXMWg6AZ+Au8/ldVrCecePGXshAu+/kiW3yKWb5Y5I9dDJamkUIr+3Sag6atrLOdYWtda49xxG49p6jjlKSb/oNJQdqZmAedqCXDQxOguP4s7SXQaxck/AagFU4FCMFqBJgu8nHi0N2fAURKuRFyQOLtcu4g2Q5qU0/sJ2gh3eBvOpvKq9gLgbONQKvpXIXacLZAPI83BBu2TiJbVDrgNfeQEZujwcE/xtNjk1RqaTJfUhLtwOupZJYW27RQHQyhotpHLsBMj2t7TQoMaJSDVU8W7v4atPnuh1SBEIcqz4NOuaRIQAh+NIxnSVJrqYXFwMC/42WFdzLYN/euMCd+W44gdXiBmzlnMFah2tnC44vydw4ySIbYfsOeGTbPK9AI7kIyf6bEEnoNqoXzTUZK0Vgnm0hQ5+SWBl2UT1Bw6YrKc5lti5Qqr+ImYrJEI1FNM/RSIaU83lrwZUMefkHFqY/amDj2uRWDvV9l6oltjUcimF/iHUNsJD5LmVzEXfBNf+rCCCfNqzAHhTC2Q0BwVBc2zdiqSzSsLIW34oMbdoQTeXvQvD76ZAFjqEK+L4VC4/ScCiCiaEPK1Dydt+KazOdRUMhscCfNRgBsFvfigs1aHhPc28z5GDLNBrlkIQ4zqeTw0eT7uM1LLPj1xn0CgoI/w7iSIo8Lb75olOWDTlAoz/rQCXXwgK0/gqF+AeRcPB9yrLK7cpCh14kYX1RfRzUZ8waXBUP1GGa0UJ9VhXhlIw2JlFcXCEubikuXdhR18gRscLg/kgmgUc47qQgqIYRMvJjzKUxYuyLRBM/YaJZskBVKXDcCbi7CkIIEfbnxYqnTNUt1DYVthguBFy9Rd4RuBfLwSPZCotyrMgHTCGaUZjMKiQFHQwro/pyPWQftTrqK9m7ekcQ98OLCM8WoShDqq5R6lzdS6IowIyzqqg7Yp0rRa6SmoW1DlACrYxtpY3HLx+uL68wvvMe/Xn97kBW2ZMZuToYA6Z2lZjKrb9rMAfLqu77zzc3P929csyfY6bzTlGLlQjhkPKDPiJshZJRp85R5Qxwr0IqQFD4DrUi31S76hEILEEUcns0OIxJKZ0MecpwSn61La/wqYXUqCgLvqIc0i0NQnqPwkhAyBkRKb0yDRlmHytIHWX0+CxxvfxAQvmWJZKuLc0CiuWNZ1UNf7EwWCz4xrAyQUTXTERB7ZwAy2luBSAPKHywC+ZBjBmnFOhkXQtQyZcGWapoGhaV9ul3sfm++lwjPewO+DvNEuA8VDD67TatfqNvtXo6njdiIwwbBSlE6q5dSk5D2dXhwd8o0vF+9OuRyriqyca5qhm39OMG+Yowgy5hfWOM9ePUuLQuo9xfVGR8mM5u3LvP7l/HxxA7LytUJXQp7li+cMWLk6026zsy3O07RczBobF/M3tUy1ilL9M1Yea3qrBGNXp/MTbF7Fl5TFXFrtfacOli2cz8gHN0HSccH5day1TOVS3Mk5yGrqoUrTIaMbywkTHns1y/OeoJlGAgzUtiH4vRFfXmJo1ziUHFmrLMLOyUUqLKTm0DnyVuIDQMpKxWc16Idct+CaxtZkV8sxS70PiWlbBpI1UTznJU4VI7D4hRv1YWuEIA9r5BwkUtVXqtKqVqhZrGF7KhnDyX+F/ARXtFlmGZTaL/odyPPDd2fqmXICuKxTRg501jmSdKtjWTNWQHp9F1tBBoAVX2VsDUD1gZ4kF5s4OKosWFwcx3urJfJ/8Jq8QcjKbaGq4kuQcXthW1D1GaQcKMJ2dHfTbAGmRVFq2e2nKzFl47en1IOgdEIsJqF8PqDgiFbi+QaWrrhxZpPhwtk74GxccBk7ByvaTAd0G7DlrVgrDGUloA9CgHKkYtDOijNJcXI4iEnsT/DcPbkKxSe/gu9jZZEuOzlllsaxoMkDH1mm0arQ2jPreW5F3nScmwxo5eB9mm2dPvTsaQySyMdMu3zLKk8WRdwGHHF8t9Zr1RAkGJhCX4z29YpXmLRRPN5jev/v77xk3XcXvgzmm83rLQ0pU5YEzCsYJV6tiAzPR1zlubvFT3qpRcNARhumVSi0YPyKTxzpbVr/2uUOOT4TcE1gm2fEhJqBlfvHXfPqgbMlCL7n1KlEl05WXO5T+DclhQPa+d4aLZjNE2wyo2m+sYrW2nTYTz2mZt4O+/rxNZh+h3qXxdTVRgJQqBQjFaNm8hy3kQoojGK73UWh5WvL3YtGK+hqesuEtM5p32yiWgqmsm65hdwVSvi7HdPklyQPujxZg0XYYDdO/SrhtNZaw2nexxjP0mufcsp/x30itw/N8JXmOrXdaIkeRgtTfExI81V6E62QaqY65TFWUULgHrJCotro0Z99w5IbhRn5FUNvxulRTUJ+px+eBVC1ElEl0+d5+yiQWqQQ01QrJHynedVVUSnzw2OzHaXkUls2CQVTLbsNk6fgKAMjhqeAwR2lmGCDUAqgAdgDq02phf7lvqYnfH68vJkWjIEOFQt9tIdLOk6JUYqCYajPN1D2WQs4jb/XqUbCRLC2N92UGiGmPqkXXyWINFGRaJmQR2IkZjrIeoLhKFQhYl8IGhCmyNREtvCMgN29CRJL2bG6xoQBop0zt8y8rtV/hqBNmiRFI7JzaKOj5YGN62F8+tkbdJ1Zul638gZdXbU61rDK+vNrRvWXPNoQUv+5ysEM6yhmTag55hFlF/ZdtUs42sX3b7mDJtiK2Wa0lBmZiV2VuVnwJqu5mjOhKnLkfJFgikyDwuWCkKZoqhIpppZ3OPwy1rd0G8O6q6niTKWp+oKq+qop0frBRsPWiWfNJkyqYoC3UOqBuSVvOghKgZinbvlILtVVeZMaRqi10sENr+BhKO9pEAxI2d0TcQItp0ze0t2cXjblnGgXHYn1kJliUa3LOAi4mGVyjZD1NWVx/uKgxSbFErwpxgW/qGwXWL7l4kdTxWzWrWYA9eV5KrG34uyJ6O0b3rFVNrKxVTGmtEn/Jrbk4AVTh1FNIFFMQFd1X7ECYyhvx1wTcExN3WFjfFp/OJHO4xSVn7lrDliDM8FDBmHXBIRc3fWzSZr67kQviFpoL3u4Dd0vo2wVv3UtdwWLFX+5sBhum1IBXxwsLHtmMjdRbZYsvDNfDoBnoAbrsh9UQTiwdPiHCNp+q8wKYv/QOQd63ff+CKo+mVfHFt9NCV7Q66v+ld5w9URH41VO3whKdhkIsXFOuQM7A2UTYRDWWQM+8eN3C3wBP5Y4Gy8QYOK5rMmjzFYLEQ/RmWFOXyVyNoQLFr8Qf1OAcmI1kRxEoqrO28nWXgCJNcWhELs3ZyzzwKe0h5K9vUIATNaSAtQip+1rPeQYqPAoyJAKNRSyyxey4rYojQw8DbIX5cpl5cYEL9CqbirCrJwqcfYGOxLd6i7uQLuLJMVrc2lZFAh0aZQUQzVjokuG8SURXDtmKwIxLRJ7trPRn3u8+wp7DUOtED213s6+3uSu4oF1V/uzPSWWB4AxhlZzF8QZ+0hxKsyYnH8z0ZYz3XaX/DHWCrAzssRstO1ph5kyUTINscMzhq5arj0i0lqi0T4Kgg1rjofhBrNUNK6KYuQwy6Zm/GIaGbOCr5fzMiY8keOao6TdsHV1UUfNuFSflh0AXS6t+9eOX9vbGs38r6Og73BtwtAdv/FnnQ8iodFI3A004SDvwAq8RSf3es1uqWgKb/qt5x38pG8+m36Tw6Hhi/mYNdyPYxUAnXS9ku48iaDBKLXabRgx9g2wh+4TLXgVABYhc3eTCez82VjxmkLi52N9hGqikDSPn2FefzSR8bQ/4ZW4Au9hIMIeZWka+GZ7rzgm/F649s1eOOmVSr5gtsWq5tHOQblxerVfBkWw4gUKhWAQv9WGVLi16rlmJGbjrRNcaUx+jySN2jsvhhaq2RlRrDXWPqB0zGSOdPluS8NKh47JJdDjju2LawtTBmjxhgza0uHuPPHaV/r7I7cXFYAQRkzhXY4V/EgC3hBgYf59VHFAcwy5x68gcXAdj7jFn9BlYpFfgqUysfmpPgoLhtXrPZ5NGSpIWZEC0NuWr3dug2kVuxEOymITQv/4y/GJ1k2B2jONnsEun1IIt2RWnIdcl8TizXxQ5T17UkC2W7ae+/XnN9YmE9AAA='),
    'evaluate_wham_feature_substitution.py': ('6063f5dd58480b0c0c7bd144fb56f10e4ee206a4e0e740ad8da8dac30a08dc11', 'H4sIABJso2oC/809XXPbRpLv/BUI9uFAB4QpynIcbri1XsfeZCvOuRzv5kHHQkHkUEIMAjAASqJ1vt9+/TEz6AFASkr2qi61ZYEzPT09PdM9/QXsn756uqurpxdp/lTl1165b66K/HTk+/7Lp3+bNKpuvGKzSVdpknk/vH0/85J87a3TukmzTK29N0nd/Cv94G1U0uwqVXtpXqdr5f36w8u30Wj04Uq1w8ukqmHI6ffvfvU2aaa8eleWWQqD1qpRqyYt8jrkSQw6/TNXuyrJRmmeNoAo/ZwgbEikXFbFLl9PmmrXXHm/vH33k1cVDfXXkfc+ufEqdQnUqsrMnG6TS5gyqdSIFgEAn3YpdjeFtyq25a5R/WUVudfAWtRtsmq8Otkqb1PBvzWtMcVlNyrHWZMs23vqOsl2CTCPBlVqtasq6CauwByV8m5S4POuYYqTulYNkPtjM6pUWVRNbRcxqctkpYDhyWVeAL0rYEleNIS3TEpV/UftvX33j3evn77712tvq5IayN3CXEAZ7OFotKmKrRfHmx2uI45h+TgBsC43bBqNTFt1SVtkfq/qa/N4ldRXWXphfvIfaIh2wELT+ltd5Oa53l2UVbFSdW1b9vaxSbeKCVsVcIp456PkYmWoewVcTC4yDVQmDU5uOt/BT+5o9mWaX5r2l/neLuW34kKQm++25R647OWlIGFrn4tqdeX8iPI82uzyFW8ojnzDM7778Scz3Y94jjQdOMa057lu/LTemjZ8Ho1evn/1w48fXr/68M/3r72F52/gkF2nTVwns2cx7DOe7fhqW83i65k/wrMSv/rPt29//IDAs4uzZ5tvvvn2m9NvT1bfPnvxzfMXz15cfDs9U+sX35xdnJw9Wz1LZt+ewZaP1moDNMW07ACPopojd8be5C/AgihfJ1WV7OcjD/5LN15ag9A2Sb5SDBxqJnxQeV1UY4bD/yoFhyj3CCgCmU1WV8E4WpU7+JcnG48EHEyV1DQV4x1r0uqrZHb2PEYVEODezmlLibq6qXi6dXqJqmdhTl7Eg/QEKD10LKKiVHngVxf+GHcJhqtk2xK8KSpvdbXLP4J8eikogSBLthfrZK4hI/hnHbzwnngn09kz/Wccehe+L5bd0hPtyjWIdUA4nbXq/it1y0+BWSyI6Ao1Q6aZW8/FFoTep7m3yYqkodXT01yipZYABvTQAGvh+CvsI6Dnz0KQpnK/eJNktYI1fBpbfu+226RKPw9RQPOu01VzDhwJeT7vv1GdLZmQTZbgNjxoUmAnbBP0T04c5txZVvqgOstM1f4cpwgQeVQDZeOwBQEllvuaLQyBLcFYwpRnUwARTEG40DubOkAgDQNA3545syW3ncmSWzvXF83B5Dat4yS/zFQMcrVNmiq9DdrGuSswyFLZwIwkyBpYyV1ZCprlMroG7VdUcV5UW4EwhC3ZLiYnofdRqRKfP1QoP4jnKsk2sUWmH5540+iMuust6E7bAUq1Dsbed96Jmjzn/lUCN6+hQm3LZh9n6UcV8IBxC3T+P4RraYFBTQRidtM/9p56bovAYVEAfd7EwHFrVH/awTUcIIJnL6IpDfuE92aVg+K1865gawL9WNSSBDjmLdOACTTn2HDPnMEE+dHiPY+iKPTmJ0wmmgOwFdV+AOZkzjDNTREjs2fRFEhtoewCIhAxe+jhWlcVQFvM0S4HQKU+k2AAmYM9Mx69qoq6PSWfFfzk/SG0DHMbeqA6PnfmABNubYkgNLyKKSwEd2DyeaBnhj37bgccvCl2fB7ooBGT227PTA+Z7Ad6aH4eAsYhCGOzb4/gXgWnsGWoWBbtbkbUAO3qOl25HdTiKBiL9GveK5745yJX/O8SmB5YmecNmgjmdfdxaLyZw6L5mo5WC4pAtGyCMMqXlQWqDWPSxc/XAbc+SG/oJfIIfXaBpfNltMpg1qDVuk8YJqJf5/PJbBl6zw0dYnahw0zrgyjZpFUN+rNWqwIM74VFqYk6hencplMtPzQQBryJUM+hAQ+6mJFJSbWIJZxlt+6deDwUpZ1azKZ1dCX08xSj9gonlap3h+T6Kq3WrZrBvQucRXY1Ce2EVoZg/XwMuuCE0SigmeH9pSrWqk5X8VpdVkrVMRqIvAVgIfMSS/BA0lV/LwAn2OWqcVtHBw8LXGHpNSr4FqH3V40jaqokr8uiVsQvq3IK0Ow4JAjM+AgdDrR9A1jJCSyFljRDVhgZgZ04iaaou0EtghUIllXZ7tYE+kIEEOx2+Fcl6xmwQ+v1BBU702HNFqQzVlUF12NymaCJGrdaoMs0ONbDfIsPXdOHWCj4tjggNc60wugBXfcMRU6fLpofr+Ih66FHXQ/P6dhh2z2HqKXJrNzh47pKNw2f1gFW8fHtdRzTAweZY+Y4yJee/nDHW1oOIngYQxzJRF7ACYWbVXv2GFb4pdmhTg/A2XtbrHdgOWjfA5gWxxhsiGMgJ9sQH1DFtz5BvQO7EnSvhRsLRZVtogvQDhcFSRW6mqBcFFgN8RZIzgLHs3DcQD/E8wdiCoKwZqs6RPc1JuJVvZjase2Eqyvw5lWGVoMzN/pksYlguORZbxP8HxgGHPhFgSGSY6NLHnS9KvLr2Tow04BYz16gtq3gV4zW+wI26CJNauN7dBH8vSp25c9o4p48p9EDIK9/+mfQb365TkrURy+vL98VRQZUBCwZPcg3oLga8AX7PT8le1Xx7DN09dDPOx0AA5YnlYQJtTs4wHJiooksxRe7zQZOg69lmhyYUFpwASF64PC6WdvRsIt2sD2b4NXeJNWajmbI8awHyS3Jrglodc+KRmrPS0B4z+dwsdP/Tmfzyels2a7BXtFrg0seqsDgGXejB2LcEx7XrhtsKtmCjNRKDPyzNUCwwPKtdaVWH8sC3Mi4DSIYe9Hwg39pVb8Dz/O8L/+h8H9f5vuldnxb/K3PBkQEnXlDsMzKOCtWpMoW/qrcwe7dqPTyqqnjIs+Mc8xOIKBJMdYJrAG0La4I1hv4stsfm/iMM+irhSfjSCI4k6S18t7vcoyuvcZr0xXkjf/6tgQkwPc7ieGr6gv4/RhE9e7kTNCOUZO7znq/+B1xIJWGRltfsY4jDIQGAi7S+4i6ELkuuHnu6+0V3T6YlDXq80Y4wKQd0/wy5rgGehIiwOCwcO7wTnj9qixWV9Dd3QBul/EBIOZSDUByu4RMVitVAnsHgG1XHx7Dbrzig+MEiBwPrE3XdOoGRopOJyRyldTqND46tA8zgGGrmgR6k8PjLYSNpUhvhg5DUwTamQt7uyrl/uYqgRuwqLTVRz8xUG5k/tGqwN74LD71vo4omghGpqqaYEqnLrDzaM1NYV2MRRL1dYT9JsL7swLHs/roeX/yyn0GlMwxSYIx7AVDTMDrxMzIpClADV2rjPV5rsctDAbh8GzLbEE+qG3SptxiGp0IvyZW2wuwFs5OZm1jHmd47dWLUwmISnmB10nbWOV5TK63/9MvH9762j2Sgvt/ogjpODuS+1Ht5xxmdMK30BxyM2ojqS5oF/xlBOK9rYX9BRoTMyQwEJ20qqkxWgwCC+yMtFLls7hNa3A4LkNvlyujGhdmR3qaih6tOhJLgfk0Jg/obXE9RjVzYqhlNCAEQ3YFR1mjXtzphy+S3MVd+9xTzOa+1etphU2Klk7SxHTbBx3JIlkxuRh26fkSlRHs9tncnHgagJEWGbiIPojNUxabp2Dg4qxPjenxFHNIIIB7XkANy6HIlkwxRdjKBj6mDewxC3ytHKqijAkPWtBIutkaQpfWZL7j/uBv2ltVmeajG7XxXxW7bE1HSss6bxbM6OGMaQPn9o6vRnvT7SjG2lkDt/MqkIwA/xnbRWuqInULS2XYgP+MO6oTmiJn4+yOrvC4UlNcFUUT4D9iL/FBWzdJvkb1TpZgexoRHreLULxJMVLf+Rkeg+13YsrVfyzGoUEulIziUJLHLIeUhF3bvKsUbFcEbvc6BQ/OTfOAr9ikeUcF1eQarVAPVA5+VD3VABY8dmKMnYsS1iCwjSv+7BEI+MusuAh48fGT6LfyEu5QOqnOsDGeX1yUe4Y75rYldtQecOToz0XzBo2+jjoSp32TArWJSJrTRsC1VlHOYs9ZOEM4cCqD01t7rRra+MhOvtY9sRrvQmXFjXeHG6m1lk1VGTYwOEqVjo+1h9pc79dwmxZxuqZkXsj5ePjppLf4us/AzzrHQVpB2VlyTOQvLKaoqkuQZ1AqoEVOxufT5ai7NzrmzpTAEXVQjcRJGzwAXU3T34iN/1bfJcR3y13LddA1ErXROaCkY13ZAN4YaB2F/uHxMzW22hp41vIIEJwvrWQZtuLBtywWeUC4rZ4/EwtjQsklFFx7CoeB576jvJ9GNJ5Pz9ZfiBopqYwDuUY54o50EcFRUpYqXwcM2l7+KoPxU++7hefM433nZSoPWi4dw9lCnTtIlnKauiNy90pWKxU/F8xJz+XFF+L3nTmM7H25e+3Kv7wVaAFaiFoPmy/HNC93WuWAUUQGRicLfXFR3Lot4LkX2Y7D0V1xOmoD6OsF/DhKgiFmCiLMlt3k8akOP1O9UAs5m58eBF0VBejbXF9bmI574gV2TSY5svQmmgDEx1mcOaUpcSrZposJ1qhOrshSBbScC24ZAGpgHPZbT/RxyIpclypg5thBxgBiOyxbmHT986nFMZE1E3azzwlpC2ZptS1Lkz9rGSYuSGMsoXFbRqCT0UpAJgZBj7aQt8Ni1hlWmwO5HUAidsXGbFHa7MYAklMsqzCEdJFukxo9oM5Gwj5+502jU3nMbwdORkjjbZqraPRSwGNcFcFvveM+WAbDYEDEb/IonXxDea3+pLrsoQ1KlCq7Trvjz08w93mypGwdFTHwsmc2V1ULFHqJmpKJRgnr+is7XWm+URXpAjSkweHB9YIxUSWrxoTG2vCd9rKodT4QnAnFrdq7ArizpxV0fkQYnvNHuQcaLTo2FDemIpBQxzX7vnoo1Y2bsKBt0LoGHGvQb4Z+CSeuMpUlZU3+3VTnpMiyQ/8Q9SwWaAkrOIGzj1EAPMWCRVgZZIkXQZG1qlcLv1szKEzWTCXXSofzxU3J8QGyFvY1OKtgP6GtkDTN3sSxxT3V0JruWadZGXvksIG4us9p6d5DYk3nzIO55sXXYoVLN0ZOCvpB0J3LlexEKpfjci3yz6hcq9hVqyHT9fKC1YxRhQyIWuca4zP++7//DU1iq/N2cI5ejN2IvtVsoRdTOlQ6ukNThr1GupFmy34HXwbYPaN7ZTaFU9UHm509f0CjS7feZWOL9IbzvpMPySV3gk1yzd37YDw+MKm1GmVeW1MxbqMGWHaaxxegwj9iOKJVWzqaU3H0BB1nLFbbgPbdoRYWgRmZe9ARYm2MOcYfTRch+d5i4fmr3Trx3TOi8/XQEdX7fHVVFTmWCkjzjKX960GCQLdqgmVJCOoQw3VDKTPP1DgOZLGxNEmPhfOop71HZVe7vBvR1DGaeRuY5CPSqRAJ7W051G6IHuqjJOXH8mAXBhYPdrLT1e9cwRpAU16rrNvbqSp0dNVc3iJOxE0HhqA5uDXFLXy5j21mHhZxCqd7W1DiGP1LuAEFDt0BPAeGV4GdJjQsEKgokMCP15TCsKGyKvmNva14rRiRPSnuzGHLoVDwQxhfWBh+WSVubNE0gn/QxefmzYg43BxNJxlX+hmmM0/oZq8G2NCjvqUmbDe+X6Ih8ikIgFWUlpC2C7fCdNG2tF3IENOFz2h0n8yXMnuhQJWkzd5AwW+ZdcF1mi5edNsJC1dVYnqRDaKPWWE7+adbzWnrDnSqIaAik2NlsZgWnB9gEHqLbLNhIQJM3JbbMt5xhy8HwJBF6Gm0wBfFen8EGPjZKVPdJmkedIoVqKIfXQ5T3R+9rC53+IbAO+oJpNmwxRx6xcUGi96A79Um2WVN/YPKyjcGWBwenipK1us40UMCfzLh1z4mp+vyBvPaeC1xAMe8eCHN6GEUzVWl1AQQTOhg/U4s+tKZtHH234sJ1fcEQ9t/CMEfp8NEB2qDIKV0Nu/T4mR6dDC/wDI48nQ6fRAnyfqboPU3iOb5s6NYwF+eoEqZYAFPwrlGfDa46PIVy4mm96Iz9Ws9vAdwTqOT+5E2KoGtqiZUQ3WEwLNjBLZiPZm0aV2gM0tX+wm4UCrzpUfBKH3UYgrkkVSkzcjGPEwMuAKZXPg/7LZJPsGXGtCzwEv2Gnx0jB/SBGD2FPiiExnk+AYRylV9VWRr46wc5QUbOr/3rIINZs/rAKrjRzVJs0mBK8N7GYYlK04y0oLiBuY2cdDqEu1LjYf+ICbMD2r9zfRSZNHNfSAYj1jHqKxCt8d42K3UdgDYpjvaO5ANw3bMfUn+m3QieHWYh2Z/yXh16NB1VqFDzpzCNkHTZSc52ZrQmjl0kwQ23mxfgyOcc88H7w447Ud4uQUaibaBtfm63aYU6bXvekW0/Jh3VwSw/MsUd82v1DVfB/jjh9cvv8cCj9XNeuGyCI4F2EJ0pHQ6GrOuZWCTenL+rxaeeFXqd9XD6NfyCN2dQNYWxogJe3lWkQ9YDKTgaGl0gcVwqKgtojDitTIuBbs81gXjnwF7Prhc4e9gteg1SAPKN1z1GIf2KNVu7EUKKfvf63DKHeP68mdgx418/xG6Whq/+BhR2dVXWoo5aogagyNa+C4d5/y7MjKWYabQPMSmmIMYIgq3DgmSJ2v6Tb94U+CuX7nTnepc9ywHanf6sNwjYW39iSjj6Y+zncvjhTyHRgqg5WApTwz7FHMhNKAJ+nhEWc4S89l3X8ZUduPWlLZYuBqxPbLHioHk2l2EfTr6VUKanHYqt/YpBvz9mqIvzrHVsUlRCQHnFF8rjda7bVkH3aMx7p5ct6SmUzzkKplWCMNh/e0cSid8ZFC7MaVD6LU8UT0H6mGO3+Ebdm6mLc3X6jZEQY1NDgh1vcp36O40KmCJhCOQwvkTMTYLvaCyJYnAiawETubycNJzfCA5blZgoiRE8FgmPe0aH6yFMRkm3wu3CU/Mp9aMEF/NvrJvi68Hk9E9pbzRt7axkDElOJ3LClxT8GOIHvUTe/y2IkeOEBIuQhcrh4gNBhFjA/szpdeaOZCZpTm9yO0uftoZjq9UULn3Qk4sYnmUbB0uRzarOTfYKHdpyBgv2X7QP/FcWQqXI5tB1pZu3IaSjsWacUQ373D/CK7tfQigJoZjUw8l5H5oMERjs4EaslMALOSSlxXzWwS1SB+QoLfNd74mF1QnxoI9U83Kv7+MXDHHDTCbdlCQHXk/p3EiBW2zZWQ36GCwGfKxnA2NwRD+EDi2D4DbfP/QGNM5MK7Nkg4NxNSIGYPaprWTjWGjRc2RFTehSJLS68dlHOiySznQb1dlkjd2VcfhKTx2ANZRvqQ32OX2/iI10dDKBXDYdrr47BBXsQ1XMdHq64ytTPobnAjMYOqfCN2FeSFZwdErxnHzSBz9NIdXVOKci3mXQ2qrozzM2zo4+fHEZgdD2C/mGMhdSWo6I+RF7nY5Jmub7XKBTL6yv8SuAvl6YdYo8nqu0rV+gEjydNIfXVHsHVpnrYeqKVxKb2NQoxxuj/kV8uE6EiuEXDYiS0UEsqFF0AwOZWPxAvR0LEtUW5uDSgN6uAyZj0SnkwCn/PpZp1Sgx1cLLRgrFZbuHyiJsCPlJxFC5jHXk1AlwLSDi+q7xRvtjz4JFotD8dEDYCnkdw77BLGJNPASsysnQy85Di1qaOoTfrOc3vFr5cK8P/lIHnSW/2BJsNRYtWjew+yQZHz938sVxnI+pXj/QW4ITmDgge7SO9d21p4OmBj6ydVKPn7Z4rbThoIDzfin06PPrD/vM1x3DbDPzSp3lvCik5OXL3wMzK1TTfb5HnCdYxJ7cnhAm5qzy9Mv5olbEHhuvzfQQ6MtuIdnu41a52AY6pt+3rebGV9074JDCi30njzhUzFsbfw7kujSwj235u3y4Un1h/HJ3K2P4lPXKvj/wydj9j+AT72jorWte3KMQovEAqkcocdBPdxl6P3DtQBBI87NHwroQGiCKGKM0fWjb+bL1YQCe5/gh+GTyxvGR4kZg4ffdO8MEiRJtaq/e/Am0rGyOt2CKqjSZh/0bdOubLofh+h6loMFPHe9ch7fwINeshZ0H0rn6uZCWQ0AaVtTBv7Mp424rf8lpTb+KVik94JTzzzeOQJH0Ei+D6Bxdv4IGt7JNn13LxIQq8eS6KyYTo0zC7U8Ag2n9ofRcG5/GNmXAc3E8VAR+JQn63xywgXJnYD9oSCOOYnd9rEzqqdR9ahuuztKB3PsFAPC2Q3ldMlxZbIbyumSwdAy5uiEc47FHTHUmBf5hL55xdFGGFvjlx3B0/xYezeqUvbTjWtfT3PQPcParyF+u1mMg8OGGdsxeu0czOfxaOi6GKSmZWvnehimogX/48re/RjgH1f2Lr7HKXtn6L9L4TtIjdattsRerS65lvJT1QRvIuiIM/y20P1TjMfuJqDGaJEOKTiHxw74oK7V+UarWIW9QIMnztTON4Vid5j89ZTefZAjQ/zi3AuxYy5tjnIloEv9UoesrLqgwIKKu7cBKFc5+3cLjtIADfHQxSHrr8xaBlAOrdOOlHMcROKkHg9dMkh8yxCJ18DaS0SmyEwaDSu5siwgdkX8Vca2MpVeT3V4WAPGbRJfg6LjVZ4MfLEg3tA1jaZw3H5pFq0NDSDG0MTQRX9l/rQtJZl3/VSR++TClZjrXea89gPdPcd1eHcNlvt336I5dgoes8sW4b0mxX17TPssj8+qoPrHDiPxewq1wiPkdz9rTF9q1rXcX7ffXKbSBp1gw5cTOEjid9bA92DM9yBgxwi3vFe77rQ1RxFQK+kuSJaWsb5MmbOd1x94ndsyi/WnZw6AbBW+cR/DVa96aWkCeN/5VHJxk/PXXcVXk//c+2iy30f07uWEP6XcflC5XUDUeflueN/aiqv+5okaEugUv8IhuPbjCvzlWywqEN/NHcpYd7egX2hxBFcf+BA6UwMwUObQPVZtociRmWU5yTBX9e3YZ6ktlGjrY/VH5DqU4OWMJrm4q4enMvJJrL2uY/68eEyfF8c7ols67NzJ44EL4KGInNt6CNE2zXd13GYZO6xwHa6OgjysvVwt1xt/4FoMB+Y2l1tvZdRxYGcvy12MgUsTP+mvq2MzxTovxLR1sitDgtS1sIzo9QNbQ6O7FltvtA33HNAHQonyt4jtT+dyRzngeA2KAyCMth+xAoN/8FuCoafATm7i4qModpEjb8COVTGWzknXka2CkPLOebOYjbGu77+wIJZeksCXefxds5m8MCVkgsZYf8qD5VR26GgdWPQOBWlzBfsOl89t4Eer+tqX3+vuItYf78aq8FzdZCC2C3+IrKEve9Na0Z2ASaLv01XzKzUEDAfecaqyNdWQLDDD7zrQ06WIWzMmZh5+ucwJTsrOqrhxHXHjVGxLfvni7o9ZVo+4hO+5gA9Egvgk2HShvzw3ChSeCGY5oAl7kRzHzB/dH/mR/kU4+gMK6w8rq/tDP61lPjDqcaWHLszjygplVZwQZX3UhCy7VZ/vaYPn+B03K5NfOpWh70QZtse2FRaD9oTzi+8WUmEJNpbwcQk2f8kETCpDeTfy8ssejNPt69u0W5n4/vU/Xr/68Pp7+06y3mv9/5UhrDcyXc1tImw5Weo1GqX4rU4U9DimCH4c41swcayj+PxKzOh/AYgUi6TmZAAA'),
    'evaluate_mobile_pipeline_3dpw.py': ('9305461efdbe5c1fa0893b25536497643d636f5590695f3f8f56dc0820379ace', 'H4sIABJso2oC/7VbbXPbOJL+rl+B5X04ckPRb4k3pxlNnXfj7OQqTlwZ10xd6VQsmoQkjimSQ5CylJz/+3XjhQRIUE4ye6mUbYJAo9Ho1wfgv/3lpGHVyX2an9B8R8pDvSnyi4njONe7KGuimpJ6Q0lUltMsfaBkW9ynGSW//Xx1Q1hzz2hNipxUdJ2ymlY0IRdvbn8j6TZaUxZMJneblJG42JYZ3dK8ZoRKquHjJtqGKxrVTUVDpFSndVOnRR6Uh4DcbaJa9S0qkrIig0Fsgry8jVj9a3oHk5ZZFHO65DGtN6RYrdI4jTLyQA9lkeJ0UZ6QNE9raE0/R0j+B1gP8FTkdNIwCj3If398/5GUBaNkVRVAi8KYMmsYXziLtpTQfZkB5To7kJw2dQVTmERJJFhLP/6CogrIuxrZKypgoSpq3mfKSuCWJGm0zgtYbSy4SwpgIi9qEmdRusU5J2VU0urfGfnl5vY9ubn9r9vrk9tfrwnI4bGosmS6roomT0DWW1pXQCfA7ZpMgPstCcNVw0UawiYgAzBJLjlgk4lqq9ZlVDGqnmO2U3/+zopc/c0OTP1Zp1sqZiijepOl94r8LTyKF/WhTPO1ar/KD+10vxf3MEI95c22PIDESF621IsqllRu371XJN6hFknafyRb1Yx/i9Ymw704cFnKl7iZ7bzPKxuywV9i18lk8u7m6p/XH67vwpvrqw9kLhgLapqzonIXp8HL1698Ar9eXfJfp5dLL6go28COuRc+OYP/Xkfkl7s3Fhrn5/+Bg8/PX4pfr2w0JpOEroQdhXURytH8eSYEE/CfHpn+JGe4431mEwL/oqqKDjB3XgYR4w9irE8S2CY6h/ZVVkT1xbkXgAxzhvrvngNDODs5IeevXgWnnFRFQWo5ccUkKPeQ76DLycL8xBAaDtYFIFfC/mgikH1cFaXLyQ6X4vP2GMyZVuF+RjiDRuPBaGRpQnsNn6EB7B4Wfg47NOHC0WYQsmFxBB5szscDs5yAi2M9fb2cPyGbVVFtBdP4j3f1+WSe37aKWe5U9+Dq7dt3H6679934lgXfaDoNTs0GJQgQsOT0vN9lMMZCV4luhIy2BNDBCB31XKzlk3gEkw7+/u49LObqU9d3lWZZXGRFNXdPudKcSkJKc7dFQrMwobs0piG4mwbdtCueZ1JhxRPfJNy0/yWsluor9+CUpCsiegWotmQ+J07cJJFDaAb+2onLBhzf5D8FvTRfQQjKYUKc3fUkIxiwwgJCVbUTblDsBWo875jNuNeQOlQ3CXA667xCIOPNzzefzn8Rb/1Og0N0h2xGMoh/C/SFS7+jfh/V8SZs1dKYwf7SJiCpx3UDe7PQTd0nx56SNK4XIFEfXfFyKQS7D6viUbELTiBPuBUvwRwWS95jG7GHZzuhGasO+qywhx8gsuo9i3wFagebovpze9N6cEkxCh0TBq2g0xPeDmYEsorAjac59/mdCYGVrSkqXkZzV9sGz+/LXVPvhLJ47vBgryuD0/XIaLSj87cRqJY2LGXRPRgFhmiIhwGrE1pVAbTW9cFVWj9r+xdVuk5zoCGXq7kfbdGCoZrGIL1QZEvP9keB4DJRHtqiF0JIMymsF30RLGeGP+CZkpikKEF8SMPDMMiKpoqp2VlfEHpM3iWArdrRqnadT//8u+ONDmABpEKQTLmqAQZC2PCGI3qSUOMG/XTqGDJhda57+RIUAX7A5o86ruGU3qRt4nKDbGrOk5wAcq9VGEOGBW7T9XTvCOkGamjnOIISMl4wM5PT3mpMd5xu1+zzHHk2vTSYyRxU//TM79FCDzA/5k09c8QmylZzq880+8EO3sNC+ureLXjM9ZoaIqwfXwTskMcbyKBxVzoyhnm/sMoYQ5PYg4mh7GqrfSl8VPzPadnqE1MvmNfX8aTe+GRD0/UGk4FWZ1BjzO1YSRLBfbGHRBwqA3RgmGujb9HfeSiC06F9dB5Oei/9JYapY0NCzFApapU+FVrYCrx/HcUb1xOZGvyGaAc/RfI11Oh7yuowhbpgD9QgoLiY+1XrbbR3B/NZbNBYhphxMGzRzbHUDEgKUqPwI0riFUpRLquryKSEh0IRoUnZPjD/mVYFcy/+ZslZh+y3YUsjANMw9+zrxvOIpsYig8ck1Hbsmqzd6zRvqCknJYgwieoIJD10cn2JBdhTl/y3KIZnnRz1zWBkMfPJ7HxJ/qoXDAPOFtyuIIME9wU5pLIv+bwcyvkrWAkNvRswdb40nRZU3IneTxv9E+rchdEdjCncH/aHY3IWBoe9vlfG+O+44J4Xnk+eka2VZl/eviXMjT+ByaKb4DINWLOFyIxC/NvQNLdpnm6brSZ4tuDDlgG8cqN9yuanNpvcjw4DrzQ2TBU+fle7wP65iokXiq4nahkoYAvMnCy7wgueOfZ3xTaCqPBBMTZVK4OV/5WcBVD/ngU9jkY8uCrOlK90la4tTpfAYvt0vlRsjtE4WGicGTQuxmloK7RqiCCMrzWOYN0at74+kf7qbGnxk0LZQUqv7Cp51i9Je8mW8IxFlUA8rnngg3UBxSH7nbOa6sa1GKqHxfP09B7Eh5IyGrMCighWt9JHIRmJQ2/1WRELrG9+3ElaxSIW2WrNSTv7tDV82eI/P/5gjG89xVECsvYf7WPxMMe9iykcM3gPSAER8NQgP4q77rqaBnTQ10vI4JWQPQ8kjPO7o3s6ORr+XVuI+BEjhI2ySczIBfoQnA5kdQnqUCl9LnKvT/p4DiEKX4FTshY6FIkQ5qOcMxDT2SmCh2KDJC6ny4g7WYxkkL0jmYVICzGn5n/5fImYTlMIaLTCPRGkeR7H3wlQmlfzOle9Sn1QqBulucEHsD2EPrTEvWPXGDYob4dEuhyBt7VygxHxAzgMXJkQwbKTAa+i5ZRy0W0nffHegDj/DUFH1l8+9M3De9DcByg553dVoyWDX1dbFk1dNuiJ5NpcPsW/vBTrb+Gz1VhvIJcPCgyBCcGzmR5NepBPaIGhNQsYc9sKcZ6YK9OgZxghNlf4HU+rgcf7tj4CnMt9UWSuNkqtsGv5YsjWWVXRljJnNoSczH6i+OfAEof1XHcgjp+wNvJkxjU2XHNbjg5GwqhtVOG+9un2SfHCm59ogZdRew609IK8N2QFnnGX1lrnnsp0/Z9MuFcei4X8GGEU5R2eU+AC6rQ+tDZLD/wQxOLafAWGSIJCVeTRFozvZLSN6irdo89Wr8NLcORyqu6w5Uwctvjk0tA7NSig+zICBy36oMO99AJe1a2bomFue0gjCcNsRf0ti5fzPcd4T7W/UUL95V56zwDm7aHZDvQsyiW4lYP6ow+uxL4/8qSDzfjxn2jKonuKqKeJO0vMm2ag1jSRGCfYxVLH0FFwOqlvAOFzWj8W1YOSd54HN0XSqEOQ70ThubGHWbpN62/B5nsrF2vtwfASje0wRNBcxKRd6OJKqcqcIcqykFZVUdmQdh1Ah3SD0T8a3MiZddKubw16lYXCmWEI1xo7z9V7YQHoNVKjaQHfuxFcv43ArWK0Sr4DcypCXuWjSIRWLRxodpYLBT21bptvVMon6HQF7Ee4fjVYdbNQiHZRmiHAj1UU1LJmyg++vp0CA2MPncX3ag6UUkff7GtEck25MBSYMbzPjtbZ714a9LohPw7wyRaDatFYjFmGsJTmhFpQc3uQtbJRv90dvxP9Yqax8IKcLS3LhowTo6/fxVn9DCaE+MwxqdGzOgNM5mbTO/fUXYKxVn84vjN5K5HR98riLbsqLoOEOr4ms7Godzjg6in9qzN7Eu+TPRTmHp7hbecdoNx35a9fdyxAxramdTuvlv+YyPi4kZgKvDgj/X1VdZOtujsO+HWMtxRFQL345mOYNlXuVlI1ubjgERdVjzsZHebyt7mf+/l+cTZbBk0ONR2lkFJxLMl+rILqO8cfXz9E6fq8TZ+/eigqVPhQzgeKdXwEbuHckorZ+qI9z22pS69zDIKFmm5Hs7muumeaO4Kkop96/D8eKBlx5atLGBFIDb3hzoC3h9E6SnNWhwhJ4mqznh4JrVM2YhY9vrQ9y4q7CK6KffGk1YfK+25plLdQlByDbfpxqR7orVDLl4FpOjKJg1QeM7ghyOMogtCjde7DXm350236sJNR+wz8+0J/b4GbHG0/EroGEoZwzAFPFmkbuQ0ohoUDuYxlb4yW+tjH6Zz3xvbPNy2j7ZVYn5ClOretoFeiLYUDBdtCxEJLGbXLEimj5BOYBhjKNb50V86HgkhBqXQ/4YnZF9SSJ3mo39pMF8u6CcyzEJm5huDlz19dmgkGbwpXENnbBFfm05kW0Cdf5yI090C3JXiuOIo3yj0MgIMvlpKYz4Y1teAlGNqF1k8uqOsuG/oDaMVgj8Dvxg8KJNCttV+bi4sUUlfDNI+zJknzddjeKqWVMzMU+hjM0FfiUUwBgT5Lf3LCDxCM6ciZFU9o61KuBAYy0b4SEMVB+boemXXZhBiLNZDhy4gvUCbTsjwOXejwRSe5MfCiHYHcq0VYez0NEI++I+5de4M44vKSsDvk5jdtK7AJdes2uJI3OG75G1e/9LCNaoSP4yxibD4Y8IauIrzv8DPNyreq86Rzg2KqIEqS7paIM53y5mR6kZSPjk94wsmLbTCYP5q0oomGWo6QqDcVpVMgMMVU4XupyBR7CiYbP/CM5nsp4ZZN8Yr1nyLw5/k4FFmxez2V3uHPUDm//LNUlEq2BBC6AIfKdUadtI6MlZHx2weiTU55yTTFkslKQQG7z2jFM1QuXx6lInK075UdmHQrPwspichUa34BTJDgv5AIU8i3ms64b4AdRN8kRAv0zTeq6uxUsddBlDZjb4X+hVJzLC/PL0decrpoQeD+nSy9P+EBj51ge1AeHF/zK9uUMbzbPyeIKsk7g/p9xHblMhHB9gByaR71vaUK7pLOTMMjuCh5qHCdGzlNSw2HsxlxoP6E3XCC30ECriTiSaHL0m+LeMqchKG4+x+GrsOa+7IqwB6YA7k6SjAUO9ttzsJZp7jRTkV3wk3iw8/XV28gRSXxYzI3ZQUqQfc1VyeJroI80lLmH7BCnZu/6GkQfjET/uPjzc27u6O5mVlCOtf7kgNk4osbSfiLleyTD1vS5Il8Lfo+Ob1MWeRXbVInHl15qxoWoCVYWAupdB+CWnfl2utht0a+x49wFZzE60ouQh5AQtB/3oaAQJHtqCpvBAgBdMSnIkAkSty+5Xg6ctudJkLWBXkT106diwhSbXmc9qzFSaloGi8RgyFNC9Jgqki3NN9uwMPJtuj4pH3p56F0b9yHlGe3VfQYtiipcYZrgKVG4e1qm3XCgVWdihdUrMxScIShg5kfgk+4+0mqoJelXmUodo/qMZYYYu/EJ2EiNxZjoQ2/lgIWhBIRmIhfmT04rSGJrVIRrYdtKtiYX30RzEyGl3U4HiCBVJOcOJ5WQz3jtmqqMPC8DLI0519tqQPtdgTHd8UE2ikm7K4KUj0uF2oknn24ahZPnEerR9zMlgFZ2Mn6fXCuAjS/PMkjsPau/ijsj7PwOkeVMTiV9kmLI8JIzjNsx7eFFi2PF93PL4fdu2AzvBhfVrj0lfquENVdVJv4AeEXlK6Sl/ekq0wQBDDFKmvYpne4Lm7w+UpG/GwX1mw/wGohwUG5NwiPnUPqAcSSP38MHn8ek7YCkcLPHYOmDb91tJOtWhwHrpV2LVAqy/YWcidhvFEDAQ+RJl3KXm9XjenwC8IgabYlG0LFX6yXk55Fqex1rGB20X8xcllyiC+p8bbKdrlwEOv4naOuvP/SfrOLk0aAyrG+tjDzNGxCz57X8/4Vqx6e3JqAvpXCS7wWbPPLM3JLTZtefscytWUJN7ID+7bP0zqDf8k84tNZdHFdHwZhYxuFO1oxIAu7p52vwcuipAMkwcFbxIyiWjn9L5RrvDwnPtCA5LL9cphnWTJsCRfg9KEb6QRMsKd1Xb7l2og4vwsRNeCDEEYds1XH/LBYsA6MrdKK1aJKEx8sn7854Re1XhAE5GFRRAgXGtQ3yvz7Yf5h84vuooUo38054wgTCJyL04rydZNFFdlRyOVgzA8Q9cmb218/9seJ749DyAlQ+M6V+kgc54DiPIMl7Gh3R6P79PkHnkfcXk3F183HvnHGHQkcKxYl0o12ei6W28MdJrBtnIsqShom0U2pDySK4wY2AhZWbg4sjaNsmt5u8LuPf0BiR27eE/zaPI8PeBMMlJM1qDeMgmJAe3bQ+dGs2YFqY0fzSKDpPV3UcnKFOIkn39avzRY1ANKKqNoyzAHaOEh4n6M6cFAjifMxd+X083NE9HtN/W3tcv2vWrdeG9h3RHkpPN2Qf2pvZcoC+WHDQpnuDKJE5/SmrZ8VNJ5aUCCQt+GAIVhesH3AzFk8MOGxCd2D/wmLBx2KwHvrG54fajFTOD+/jQkeFr//I/2jPtljlUKSg/WoqyjBNHlcIJQ8d5p6NX0t8+mY7fjJvgAhWaADmLImRtMzqKc1JBkNOMa96wRAQJLi3w8qeuITQgdRxZw+QrZM545jYYJ/X1iDI9t2+SBnHyMJEAveQOb6G29wRT/I+FKaJZgOsDm/3o95BxYmXo+CkMOGRolxUKy/xKFul7f0cpaxfMUCS4/qiKEXgyGgQeNqNT7me/XSoCXUidte5Wob7NlO4DS1kCPUTnujuLg9f/FaWBxquTDEfQxDfqAThgiSh6E81BGI+eT/APzdfQOIRAAA'),
    'evaluate_full_pipeline_tradeoff.py': ('4c09f9d1c407f77b3771764fbd7ca24801419ace0e7b3be2aeac78fc6f9556fb', 'H4sIABJso2oC/9U9YXfjNo7f/StY7YeVO7ISJ53Zrve872Y709fZm2nz2tnuu+fz05NlOlEjS1pJTuLm8t8PIEiJpCjbmWnv3c2HiSWRIAiAAAgC0h++ONvV1dkqzc94fsfKfXNT5Jcjz/M+8LjeVZw1N5xdvrn6J4uTZFfFyZ41VbzmxWbDNlWxZRXPoCVfs39+9/oDawrRIb0CMJzVu1XNm3A0+gj3+F2c7eIGWlbFfc3WPEtXvIIb2Z7VvIzxJ2vuC8Y3G5409Ww0moYCmjlEGTc37D6F/9KmZoBHmqRxxjZZWqox0iL/y+iCOtfxVoPQdi52Tb/LZciKHNDBft99+PGCpdv4mrMNjxskRcXLLE4AyoqarNO6SbMMbnwb183P6UeY/i3P/8LifD36ioYvq6IscGSDJMz/zx/e/3D3NcNngeoesJzvgLoZS/O0gUmlvwrExpKCm7SqG4BacU403NVIVl7tWcmrusiRNcltyLBxXJaTLL0VLREhFrdUzvYjmEpRNYqkFd/wiucJ1yDWaX6d8YkEfJcCywO24kmMTXBmIA3Qp8GBRuuC1ywv4KKuC2AHcBI65A3MAh6kOdvusiY1gIXsdZbRNGKg7ZonxRrwQd6MsjThORLtpw9X7xEmB05/uSkqjVuCU1+KmdVJUcmuHcuvXk8+XP396m0woj/s6mf4TxAiSUAcKoKCo+KsBVORGH+sSdzlSEVF5CT2jWCygBx1bSedFHkTwySbm7QmlDU0aWKhAPpHTV5xrFE7CKJQs195VbAEsK9iwPR6l8UVu+MZULTZB6wuYDQQoWJDqwJJt4Kn9wyGjUf3RZWtJ9dVscuRkCAKv8AyKoCVHTYhruzRSCzcKNrsUKyjCMQcpQGGhNmIdvVopO5V1yA0NVfXSX2nfqZ5XcII6vIXYK36Xe9r9bNJt5wGTApYK4kAr0b8BpBteBUAlTYxiMg6BXiiMS5UUBCq4RVc0oNmX4Jkqvuv832L6i/FCnqoq3y3LfcgOywvW1SKKlFQ/rXeKhj4m+4CBrD69k2atBjiOm1HUCos2harNONRmZagxXIeXa7LexxL3sd2vT73N/E2kqokQjUAq2MnRAQ6qvui52j08+sf373+/uNPbM78EYN/XhnD4iEYK77OEBRoLy8YepoXRoMNaJi7tImU+EVpXu6aWj1OS5TuiJRTtC+y4u5reDYefXf56kP08Yfo79OvAJfFq4C9DNhXAZsG7CJgl/ADbk3h3hRvwt0p3J7C/a/hz/lydPX2/c/vforeff/m3TdvcToL7LYcfXuFF5fnMFdgPXI7uYkyfg32Jaq3ZRateclBjvMEFIg/ZpO/su8Bwxmh63mgO0DuTT1xN2VlmtxmsJDADGVFvAYdASxZ8ypnV8KynX2/217tQ7EIEFK6EQv4Jq7jpql8KdEB8655A5KPV96YBhXN6XnYPYVJaDc3uyxTD9gfUFT5jKXXOainBQ4wgbmCuKyXAiLo9xhX/Zw9tiN4q6LIvBkIbYi/oqB7kuYNPID/tXsbmCXeFX+1+0kBNOQP8ET+0p4VK9QL8Ih+aE/qpoLb8L92b5enqL/0+0/if9TGOaiqgKGAc1TyckJh2vAtME2j20Y0FbSGdjC5KMKlHkVdG/wHwif4kJeBDnssGjXVvmsdRbS8osj3khtc6h614g8JLxv2Tjx9W1VF1XX6A8lJmJe/gi1FOVkXAqecgwwRmJCx1ywBb6HiYG2wP9gVMFQrULhxSnZCA4iy+oDzA+u6Q9sADUiMWVjeZmBrml2cgUuBQ4BSAWdIdS7Brknxr/d5clMVefor99f8DqR6RsoqpCtL/mE4uh+ihLH5HBi+W8deN1PqjDdDHfZYjoc6ee0cKWBFKU3jDJWrGLnZgQgt4CogQVvSQH2siQU1LA50LeZinBDgbaKEFL1PLSpeg6qFBu1g8sEQyIqDdsxlv8AFl03UuHKOYJTBTvMcVW+9227jau8LaapnoDfqZgFSmK/jqor3SzFLFMgFCjlNkv03rrXlTNcTsn9LZiER7EdAARASwuaD39xUaSK8ZXB6hG0H88y3ZaNEVE5GV/khYYjTBrQ01CXKY8W5+C5Os3gFNmZTwQKpfbjgGUypwx74tAwA9zV/EOpCTA7+EtqiG2ja/Lq5QdWzaCcD9yS0xS3fLxcCApKGTds2uOjhIS5i31i53m15sfYC895qVTzY9+Ss6959MFYlX0cuOOqZE558NgQXHeweXqBm+ggIwqT64MSupc60bfzgb9PcN8g4Dti5YpBlYIlKA0wi0WoZFXQMqrUbJXjn6YOmf53rdmTJMD38COaxqKQQA+PKAsAi2w3ha4pIeEuK/TQieyG52oqCgIJMOB2AYJkJAJhUZOT4HAHjIT9V58W57P8QgUINgBP1Lf6ygYCx3eLOCWRC+EOCD52wtkQIxEwCDZ1Rx3Nib43+BcAXf31wbog3MK/pWPoPsE8Dkb1c21hURdFE4JyAbgIdlRQdAicSroXcUQ8ooKEoG8DAlsbw264hzO0GPEN/MgURFXRDKsIu6CGt5+caHFwlEd6NYtz1oeoW8oNescTwkyYgVp8+gRD8ULBZpvKACQiVe3lhLb1xO4MLcC8vLYRtmm9j0LwPEaBUyW1M9Go9gHc3V2xPPX0XKWxEpuT7AjqvpAygKwhOs9icR8j3T6eeN0ivjkRuyijZEFOQSPy2xOlN002ZV4aN0xxbdEZ7ZCGZ1Fbbchzu8vpfO87BFJ6PgULKD9B8Ulz6LmhKJTwbYGs8Zgc49wzZb+F1rDTAWCrGeDbI8m5Z4D9jVt1d5/SkQnDNTj5yjKkBNTn89dcnjSdEedat1gNNUZpUU/ztbpqAMgehvONZOxGMl9SdYgbpC6R1nBv9n6RxrnZ5hLEiYmrOm/uiulXA8jz8UKx3GVd2Ge33bMikSmMtGT0zHoLriM46LEH8c4ppdrmEiKywYB3GGtZz+bcj0MOcUF7AUlt2t3FRtE/EytEeKvzbBp3kos+rrtCDFTMCMW/DoXWg7YuFFLVQlMAtrSYoB2YjUnhWM5QBs5mQEK1ZJwttO008ZMPOZ4ZNxjWPlKsI6xSZgbsO0U4QGQOiMxFmos7kRhxjP0E80Oo464HM8MhvkRirXXK9r0OMgnXbDXUnTPOaV41/HlhdlfNSbGGLswrBq8lgN74Vf9sg3921oECN29dyn+HOACPZuKmYU5tJARRN13zSFLDxA4qONJIo00tXymSFmortvI7WUGrEak03XT4DgOIjxczVlWgK8Np5aUulQziwbi480c8YV1sWGqq9u8Ndh0yfVIU6/mqOUVz7JiV1lUcDzcy5urpKjCx1hyEwiqZhECyTlBG/o3VaUXRYCr1zV+EW31ZRSgk+FrzrhFJGS0gQRSzmuTIoN+Jg5Ds2+9aM2BnbeAg9erxGNKpwB3yr/PEThmS8cQjeDYZ//LGxr6W2YmsLejKDXSMacvVLnszILfz4YAzA3FgKTL5/+4+PP75+jwgEYubRh9fv39IlnkmIW9++VTfFkYhnuh6wG/rXLsWDDkDx0Zrzk2f5CMTx2QHmYdyxC+WdOPtu3oQA0hEAPY8DJox6QW0QIexjko8go9abo67rRh2b+3eCMac/5qOk4hgEggnkdTb/Ns70RT0O0dj6umNo7PcFhrCe/p3Il+bypEwsKZAgEb4TS0zEfeQa60zMEQsivWpr02G6E0ZLEbtwPaeZa2GC4xJA7X7B7TDYj2vQKwDNBdsdcRC0vdnlt1EN220ZsdACht3AWqjNDCFKOmlWwdxEtzsVbSPSBVBbIrdGpG85yJ/11WbF4nI3vqArIKADVXpehzo9b0HawDRe2jNx7qi6aK1TCHqTuTwyGV1CcA+qXZ46BVCvGAY02TJmX8zFbQ3PZyjAjXfV0vSMQDAKnLFtWm/ReMzYo2PYJ6EZbS34aGNiqD5pkEBhVI0j0IsBT9oob8tfwIQebVbendKoxZvWkt7BWPNdFzmBk9sfj6lLFU2mXqlk8IuV6jbGtRpJBdwaAzEa2gJQmdcc/UwHewJt8WviUIPBRshp7hOYF1o7J6CxsaIjMPzg0aOHqc/IlKlVsd7TbsKEtRBDzhAHkPPZ0rIRWbECeEWV8rw51HM2tXqKRTS3tIXWxWqOAC9Ae/WMjb3k27nqfBmaqib0z5nnULeBSepq43Nm2BELnNcGNIzSrnLSobpt00RrblCp38Fad3UbeAOlst1ZdLTMXOBAcJhPbvg9kNYMpDqyDB3P7lLN0rTYL2YBM4/Ll+GWx7k5jXW6nYMOv+W8xJ8fqx0fRrsdy5jGbzmQgwW9W5Pe7IfJa1xPzHkcFy3r5vGB+7KmdTYH7/YKanstLDv+FxkbbP85mxpj/Yl9FB7Y7/DMMt2mWVylzZ48V9DL20iIX2DbO5lMcJ0LA3EqGJPdNtcCkxcu1ksrG8Yl7vZMcH4v3kkrB6gSZ9ewlHGfEOH21VeYT8wRAyGBk+m4B4mEdfBpUu58x22Kd/YffAn+0Pl5eG48cEVcNY/hc6fca+lcSgME6XX+P0Oh1ln6nenTX6Pqzv8DGrl8RUUvWwQIC7c1MnsakmJ3Q19Ps7yBaVddlrBnykYH0zN4Fpf1kHeqZWYcNNoJbEic1BkbVHdbZOzcp47Zs7e5IRhj9tc5e9k5sEZ66Jz1jiqlvLoF1e9b9cnFEkhwASLSezadTaZLcJB7Dy5my76MTZhv2fEOtPlAwTXvItCgB1WcBV+Y98e0OMSj6ZjAjUxx97+9+unLL3tHU3hAMEhKIJ7Ig8ENxRrPm+b2GZeMunR0fbSyOeKI1PB2S0ly+sG3MkjWJL3hHppKtzuhLnMOcjfQQcy1y6+8PN+UeJ6ok6Dr8dT9lGvHPLLYxrB9srK+RBpuhZFumZIbvq6ud1tw8a/EE1+PZoKHKlIPsriu570Obyjbtv6OZ+W3qrEWwqahwni9jmLZxfcmE3F7PcFkVy8Q2Y1zih2ryKTwEg+CEMnrEwAwEUc6nwilbnaYXT5JbnhyK9I5PhUSnp1M8OzkswB8Ph6Ucju55+n1DablfiJdtmU2EXvISRuL/VRYN5evthOhOSbtDueT8YKWGDRtJ4ahQpXzrRJRBvrS0e4ndMQt6kQ4wxMMPTghKL1zRM6OQHn11XGuHAFx8fLVQRhkqj+V/mCQWx44QMnjtepa7KMIhPiDQPAAR+pmGi6i1FndMmIz6rEWqfCB+USSMeqWidWAjrmHntLaiOTasB6imEb2RtwBHdc4O2MebNzOKOp1hvfDcu/ph3fbtMbCFwy84XEDnjuMRUhMFA5hRMykgjyPolNZday0VO6GhDbT9kyCuFymisrB2oMdAXPGPMwhCZgX4rx8CUQd7sq0azqp0c+WsEd7stRe0JmSZyVjA8o+saY9HbRObx5xkN6BmZmubcymn+DSzq+DJY6uBgd+8iz3mY4Yiu02xRiZnvNd71ZlVYBGqQE9ITnSm+3wWHjXKYq5V/E7Ml148d3b12+8ZcCS+/XclA5YEPyh6QIdY5DbKi0751Ei8oUV28dCnuibHz58ePfxOUHxtw9YNaAqsCTsxyHITwFwfpev2SO17J33yTQCc0qhSG68A8Y9P6tgEJJkC/n/rQNOlz4loiM2Wgo6nmiovGXwakQSiQd7FE8dwsDq9TfeG3nK9Eiwnjw8K9rVN5p+E/V4zvy2rEAXLeoa0ASEvxGBSqLEom4WWl4ugKLSoRAPzH1bmY1N7Sf6RLd8bxVuWInLdvKzK/F5KOl5KOH5ULKzljDqzMRyQHH2MB7aPe1rO5Hao4ME/c5daoLvZ1l73UToODpe1VhXOGe1KFH0XaSfYK2ITLwbtyuAeh5chRtPeMtrKvJL61bjd4WkKc/WoIUfCdqTJwVenFRJ26DlK635g5FGIO6AcY7vI1HnCJPFdcdhJyqqXdtcT6SMtssD9H1NvM/EItShjMOqLrMUDHoEK2OKKZ9CM4P69PVMdanfFbqHqxW+L0iLS3qI+tGa+mINIZYZxveyDLZzKTuCC1WuvDv2V3beDScCAPIkyGxHp0AKQf38p6hTqg+cy512XcZJewDV9sCahIAG0HaSoEWUJ0bFRBkpWOCX6rlATaNGGS/JsstL5FKLwNKxm9UAKniyMAmIRvGOOlrtiWHQSBY3+trRWGVyX2WH9jnbiZKontWHU/kdVGmYpdfpSuRgPB4TSqH+5Rx0sXPiv3gGsl1sYD6XxSJPhiiauD4jdwYrb1uyS+FU1bZ5YdUoCwkWyNdUs6sqlE23wjMrsQl8TYXfeP8G3Exew+ayyDdYxAxGjgYQFbhgpBNRJ2tDXXFVh4zQ6wagxdeAZ91Q8XMJTJIFtpjC01ZHh73ja+kuB+oHBu5BHQvNY9k8zO+iRv5Rl1tabE2GZAqpE6wj83TIIwjcTnx/POmxkZF2jmq0GHZBRuoIsj3IxjJdv3VbzB3DWM+Igra9tLhBh1Sfo5kcJHcaUS4LMDrPwbUhGS+0OtqAzZauxBtX2YA+UJe0MZDQTtLTFp6ZeandLzuPYbk0lMddXKWxOLrqSrJ97EP7IPkYdYkqUtYWPKxjkFMj6UgDo1XXnTKoaHjKqFj7rIyLTKMwK620RArUHhUWoq+jdZVumoM5IqS32rcjRAO07VEUQPSoR/SB5Z+p6vG62FWJKuSDLuf9JmveqIp5fD7q6XOsXfeVhkRi18nc00y5vvFr3ZH5sCUaGeUJtcirdVYcygpDw38R8i87Gq6AAbAtn6sDvYcBSTb+t7kNBdmX5tpRtHIojxa1da5nf7K4/ekBUBIl9xSijWmeOnctaIkbdBgtZlrB2NJxFiTzkymDHathrdpBg9JdkYPnmdpIz1h+NizVUYNp46eOieRlzYEJgt5URmxmDcgq4izertbxrKu4UOUK5qRdJ2QKIzWsuv7McU0KuQZus6yP5lopNLv8nWM1BsdtqEVskwqHsDXI0j97s2Y10rxRchHaco5W/DuQxoLgD6hTmsjuaM5QeS9WpAgWz6KXqHR4wZq1o4spM9aTNYDuOAR9kiucKTlCpKK55MdBZ9VTMab1rIqKnySTxr2efPbCZr26Hqe26D+2uXK87E3M05lM0kmvGeQ49CaSmb4igmNd1OtJZpbcWzXZA28tmdlc6Q7WurNyckUWQ/guF17r43pL9qJ30mtquxc9PfTCXoKuPKxhJBQF+niYI/dhDVFl2T4iONZyfhYgCyWX0LfwCthHVXeUqQ1apukVupv1cLqd73a4Q1tEzQ8DsyV/YRlbe9Hprx4iveN7fecsqmLpt9axHkg+aTcbgfO5U+sZ2s/9iIKNCHxIMz1LjR1SPf00mLGLyvLVY5H+toAux6Kf8KBXYr6cqnCM3MnIs/2WVw9Uf45ZN1aqzIEiUwM56d489hARBcbtMGBqTlOAekGxJl3PBKAVEFtS+UxAXanuEDtO6C/rzXRBV7nU+OhYd1mOq3eX4Yo9RdaH+39GjW5fg2t8V8ZXXh63vaf5hIZUHVobmkFcuF+WtRTZ9BqyRv9W5br7Ljz8JYthUd/2NFm/flwE42FPBhsinrdWyQxKLZ+HhGE9TkFBGZDfYnDL4jg4PXIVzBx82cKzX7hg+ZjOt1UMvLFiKOHPrsr5DGTpuOV3xVaWAZoRAnmq00WAs+Le2AuBJZcd5SmjrD0zhnMexvwjx9dm0asoRdhCwpF1g19UTyLe8ai21+1pDAXk5Q4dX4XoDnap6M/jkxEPl/GkQFvYIvzfLfP+S8yEOqIiv4BK/joN1K8AdGuPvsrURHnwoeB7/6mr1rELdPbvD2ZJHPMauvjokN8xdq51SWVY3KJz9xrKiF6KKX1LjZSmwBT3PU/SafnlOGByFF/7bZSsYCMVqnEoNEo2UsvqkGVCMSKOR9qr6uhlpyQIbhGyYrQdkTRgS5XWK9/+1QMApDHai3p6DA5TB0rfHI9VyqsEQyfvhg8uV6+Kr+Kr5JwK2jGHGrMFHD53Si9F7Q2j8Pa6oKawmn1DY7Q4AiPCM9VT4LAzyv5FX3x8aP4D+0VRjHjafuVzxeNY7PlkUdED46odkG9svXRR06KtOAJB4brTtSJYbkeEhAsinjhjQNITaqtz1fmFSE8PXLvt0zqMe2n4dlRfTVZcGHbqoLyarDgQrH8xHxKmofD9C3xRaOMfEVK9TiDNLariC3bDNczcYWIenUJ0mtY7rvmsQI6ZhW3Kj7PfobiRs8NyYaR6LwfQMVWVWoziVULudXoYzoF54dtd/eG5ud+fq3sd5mYXoRkzHB/GTJORGRvGw6lzjwn/wU7CYjxXQ1uBRbf9tBp0mV69iKTKsdRP+Y4lteRFPhG1BkZeC3it2nvovTaxFd8qWrt8je79GI4TSA0JgqArTytoKnU+cK+/UunZ7MAbS8cHzItuWVx+hTIzpitjBVpxO9zu4ogQvu3Fje0oq1jQaCfk7A9EWtX0l/JFQuLB4a5afNTs3el6HcCBWKbZnVaU3nVoR2r2sw2ywePnsNDNukP23mAh0V4YNjXE4HszFApWDYttLVVCheAB7B2a2ExtNPWiaGbpZyRWnHtYjXT5MvyT/q6+Xkdnr5evwj/riYZ31mB3vR6v/hR+pfWg8pttJLTERU9LSjjOIh0DavinkaWhiOCruKZXvVe8rIr1LhHipyXHrGpfI6BFnzEeZU+18kDxHQK7y3PbK6qc1tqiEPU6Dy+NV1eJCmaD+XVyw7cxFgLWIFJA2Kn+gqqkEC+osrTdOoYNK8cNmQfbzbRueNVmGPIaXwlhfAZBpSDiFzQo5mC/nLflGMws0sSXErMimeswE06ZyoYYnwij4jJPLGp9IIek1jG+R722XZFWeYhMNCc+lgsydgNwYEFPnoeG7qWCCkyy3Rowi9rvifBKBGaHHNvey5NL1ECoSsACc6fv4f1gsBJFT3wdpNPS6uMoxjcPug+NeK7Nukz5O+GzJvgRGP6AX+VIG0LBAdD8wAm+NbgdwvGdk9r5nRMHWJlJCDhpaYQHMggFwjf4TYpCZQA656++ooKEEuai9/UUPW3Q4U11wt7O3M0889M6FY+zF397++Y9MrJKblJ08sCsnOlpfe3SrRuR7gisvYLt2pn4YI6LSHRgLT+oY3zYJmD0uZJJU+2aG/l9G8qnQUoJJ4EOWYFJhydsGHH3XOmTO/mE3h9YAh841SN2398RH2ARaFDdGegspqrvHVNTn+sRr7IjsC1baRqHvuhyeEIDITPnzN4b38HAr8HITxEAr9Bo4fcwaLHhqqFPHCFaWSIcDCHmjumRg6fCFexdo/bKWACrPkOjlodcC+qTKAzEFmHnyf6IqIrv1kTquzUwQeuVMo5GB7SR8xM+THxAaIANSApkU8j+ASv+7Yc3f3OqJMx5ZhdiOcaMvpzDhj+5g6/fAL8Zd+yHZ09e9oEJ6a450n8lc/P/I77GN3Wv0/g6L0CvJvhhqGJ3fYOvymGtUharrv8eLQFarlz5qlvxzsEkRuAq01myO8byBVikxQT+KK6eNMUn3acTzkjn4AIlI/xiknL4tSLankNBRgr9CaGnvvn56kd2cX7xFfuImYn4KRpUXUz6HTUrYTfJSJH1PghgOLOmt2p7rKZXanqmpv/p9kFf6cB1WvS8yYZ8q16qTV2LJeHwPm05KrDKnPT8o/OwUBQ2qEMKmgP4jI4geG8a4CQGB7ePW/pW3DoCWd7VkeC0cqPI/RwQiaq4gy2JC2t1HojFbvQZG/gRuJq0YoOvR714+QpjWEYep7hLBYzuow0LzmG13Eus/+Rh+6BOGllVA2hZUOpWT9TbIrYTkOzVvVnQzKT6T561CebgjF2p9J88rgvYwdHBxqUbzHdc7Rvujty07lNkiRC++MEhWSFGz3wsLB3I3fGEV3IZqWiGYqwJ2j+eIDQ45uiEQJ1kdU5vpCz34sxacUxNzeTjwZnZ6sKaJH6UENQSeLco0jvHW1XaFhj455mvSrLlXVHYRBMPO2BWyGvAKsnoFg4tA0LGZ7TaXaLKjeibJrUGOZrOKurORDCQpodpbKXZ6cqB1MiDoUKJmDwL0mMYEoD95EAMkYrJdcsYaHYwaK3eYBBRp2i7oVWGXpuiuU1unfXBYKkdFbMF6VnbbQeSPR7U4qMMp2CpWCF36X022Lgf5Mfn8MJJfNy2Q8PkBt/gGd195iT7S9I97V6zM/pi0hFSBGzKJ392vfFoOvhKrd+Bau2hhiOipR/ADJzxBS6ftQvnnBx1ESfag6OM+uQ9AJlN+wvgqX3dRyjfrAlKE3Xn9hZriemiprMYxh9S0OnFrVaEr/e8r1J8tTZ/aHzteJLiiFRskjfzizG+0OK/cuADcL/AiNTc2zWbydeSH0l9J+pr6OVGdagf9agaC9x36QOnzQ1ILWxzHnwvBAASlKjezmWp0aKn5/W68/Zw1K5PN0raNc677lL+QeAOlrvi4P0ody+KbcWnA624W3zjVpErLEpQeB6+jCnn97g/mHsuGuMHRusGdkjb7shKMA5PbwBY+CZNmn+KGz61CzQ6zrufKA6gbOtYTH3u0acttUVFUEkqbni8NrLE9Id4purrbNZfBTE6duL9ODr1dHk2cDKuFBtFeo9psd/CWjocoSM1DocxH3YB/lfRd2cLHcH9ROPxe+JNKoo+a+prisXlDutyKnuoJTg+6Okq/ReMHF6ofeKN735LNywSaT1RJHKRogjfBBdFMv+KXgs3+h/b+26ttX0AAA=='),
    'finetune_fastvit_wham_downstream.py': ('a5746c88555b9048233af272c4ecd135bde46188ca02ebeb9bda915281b86c04', 'H4sIABJso2oC/+V9bZPjttHgd/0KhqmrotYUd2bWu4+tWKlnk7UT19mOb71J6mpOx3AkaIZZSlRIambH4/nv1914a4Agpdl1vuS2XB6RBBpAo9HdaHQ3fvub54e2eX5V7p6L3W20v+9u6t2LSRzHb+q7Xds1otjOiruiEdE3Rdv9rXwXbcqdmHWHXbm7jrqbpj5c30RFtGnqn8Uu+vufX38frepGZJPJu5uyjeC/tajKK9EUnajuo1bsC/yJFbZQX0Qv3vz496gTbReJ26I6FF3dZNG3XdQ1Rblro3oHterdBIvWm025KosqAhitWKuqWA470+6rskuhhUqsujZa3YjV+31d7joEQk0B/HJddCWAU4WL3TqqRHErWipA3aBP0WHX1QeAsY42dUMfYeDQ9Ovnf4ARrcoWoNAg+90yPdqUlYhaGBCA3x22oilX+HH1PirXbRa9Le5oCJN//KMV/zqI3Up8AzXa5wTgH/+A3u3qjvrbRjgF0ItGQHdEdKB26qgRq/pWyP5ti251A81Oym1xDe0qkNHVPczIFvqGXVoV0I0i2tetgB78UMsh84bErkOAagwZ0sJkQrOV55tDd2hEnkfldl83Tr3JRL9rrgkP+nnV3uqf/2wB8+o39PZG/96Xq/eVqdDApNRb/dQervZNvRJtq9905VbIDsFcFquqaGEsukfmlSyxh1aA+vTXH7FRSXn3e0SHev96d2/6r6hQ5Hc3xTbfiIKGDN1ou7I74Fijoo3oY3FlRlZfQTP6CeZ6f4+ldnvT6bpZ3TgP2W6XbQ67FUIE2oHS36iu4Vfds92Ovcyg+arNcIz6+xv4/V1drEWT0u9WdJPJ26//11+/ffv1m/zd29ff/pD/z6//90/RInqYRPAvvrqqP8Sp+i2ghn4AKtU/b8u1/omUon+/31+Y9/+klfXijX7eNEBZua0HhJYTlZkCEpPm+VrsoNfw9Gj7+7fX34309mkdBPLt8vf7F+4LXtrvMuvh42Qy+W9DS4nkbot3zUFMJ/QqeocL5B0u5jnVpnUNwOYR4IXe6AU4BxbQ0Jtyty6BkudAF9luXTRNcU/v5aLMRdPUzTzaVHXRHW0fmMdPugEGpA+c5ngedYd9JS7ttzTKsmxJJeRMmDLQf/VxMlmLDYxDrHMBXAaEA6yZBJ9plNNo9ntgITvVAbluM/xMZab0FloMf5AUvS12h6LKvW/lRn1eHdZFVrZ5cVuUVXFViWQqG7MQqAgDkxdVpUDJ/hPzW3U5klKCixtnAFY8dd8iRMLFQkB9anlnXZ3TYlb1phkIwfu9SKAaTdOLC9nhRgDl7Kj2JaAujS7P0ug8jWbnS43Grngv8qa+a3kf0hBNUMfg41zjAoQNyOICplrVTdXY34ldWzcMJaofstSlLIT8Q41CNcZGAdP46vPpdMlHAa+Llnqih32pKi4NSnddeX2oD22u6F5+T2BdihpWAcNwVbYdo7ul7KwEHMS0BjHNYC3eFNDP2bkhC5A3ILB3umO9kV/KkVzVB2yvpCZosopuV+9+Fk2tql6ez5fRbxYaVXOYqWn0WXQuVwSoNXtVldQBRBXoLbtrkbDW02iNeFwYPKas4aliAmIHDEB0uKyWABF+JvIT6hXUENCAapHRtmInUAMq6j5Tqcuz5XJqCuJS0WUBDrVnPsplWbYiegv6DMjNr5HFJJv4nVZRtDoSPWgoj6izIZrtJMe2OYSfFet1oos79C9HoYikqlcoRjUbzEmlgBVQdwn+b07SmEgEf2gmBoQEfI/mLbGTC+Wj51HsqEkxviGgioGPlzy9LupkT6zC39vZNaPBmbFDm/PJM6+Rza3LJpmSUroTH7rEfruu6qskfpbt31cxUBky3ameJ8uBvbVgqk8sHeBQfqi7b5BMJTGYmvEf60O1JpCg6oJeDCt6Z9TsgIbqqI1XoqrvothA28QPiJzHWOFE00SxthQB9FFUNUoU9cKjC+Ap3SUIz5RLu6WmFKo7HyiEGsSjmQhUA2l1gMpEgoa15+KWcZS7Eiphzazew5qPm6t4imqa3BW5GN8X9zg0aFXqshk+JbJkCir1ql7DalvEwIbK3TlbT0pqS1ajua6CdsmUqCXjNY7kMWREcIDLlltkay8ioj/5krjo5exCcrzk8zT6fHoCk/grEOEe9lJAAqYjEcFCXD4gah7n0YPTyiMbG2keMDJSKxKnvZ6ICQzvcn5+tnRq4VQqwQEdMFiSWqwtansgNZscsI8SliSRYsOXtriB6gBVSlG8dBEVkMbA9O87RxbpfxKqkjEZ7BqhH0lctKuyZGgifNTNFjanPwskISAbCXkKZHQnmsQtyweVFXugzXXyEG/jeQQ6RwxghPq5gb/n+Feol+ePl7ahpY8mO1W8BYfKUPYRtqdISPikqp5CTj9COdg8baE+7WC2ZUtbVktMzrqg5X1J66/txBYnjS3wRFFdKqks1WNwFAXNIyaj/YJNsNkrW45Gm/cWiAMYJfXQYRq6q4bTUkOKyW2Lfa73z1I/aiX1gwYrqpazLFCSlmkU4H8TqzvZnYaiRdiRv1X7fm1VsJYHaVxAPtUUd5FnCtgB0tqMdvQMyYDZE/iyHLDRzkKqmxzfJe3Oln31TXYytyxvEADje4Mqt4KmN3Gj8MxOzwEndTYOTHOsQUCK1RzrlF1Qg5DU9neoQ5Js5j0KsKzLUR+PK+TTYd1SFwmpl/umvhI4F6UU8ySnDrsSCKPH0Svgi7DTEgkwH+INCA7oGJkQDH6xLXfJ+Sv+ra9BB5i47ALunKB12UWvV4yXM2rwCeTSArIVjIRbeOQ5WpoIxVYhqjDYSyNHcJk6csYV0l1CCWGerAGaAuTOnEgtRfmwDMgwvVpTufINV8pKYJ9t4rFomHQgme4+Jx5M+naxu096Mkz3euEPpFeS1NY9bMU/0G7pqiVRAdPvYG06jb5aROdi9rIHAAeBRTQzxzH8XO4TGE6muDz+dDh9n1wY/++NsW4I+YYkqLPQo99LYYbQ1fQHpDmtsd1BOB/qq1Y0t0Q/rPalacFVYGhOoShNZOIiy0CyCJtOp/3qRuYn9Gilx9QR1Th+RULHRTOIQKQZZRy22xYyfbtbRbtPjH1azVC9Trj+13a510kYvCx8yZYIdNeWjX5P1OHp1424LcUd1I5BocnQ+AjdfkB59rh4kIaz7MX1Y0xdVm3iV6Qh1eD8xXJ6DBV9mmS7IpDrw9ggM7s1rv8OVBrZ58d4gFABI0VV3efiQ7HqcDlTfwf7j1hCOpFImjGMwYpClH2x9FU1p4UpInb+EQh4vb2SokWNb4bjQ2SQBX2UROZR3AP4GZtCt4MDeJKiUFO9lYXGBJEy1keMNOXIcRQ1CcsYNpV6g0qmUtFIYWJ6WOqacVMrYRxDq9XWsJLS0wx82NYj2zego+fmG1dXeXljDfCV1/4GfhN/X7Ytor+v8c2Z3grg9JoFpOdUUMsX6nX0C9kSoKv4R+kzgBmnjKN/QINGviKlWuT0JLKSe7yGnWLomqA14KDsORCfRNqDrjI/e7l+zP65v445rcvqiDJU2n1JR2PQ5COLuqTWY+y40gyG0MzSN7Gw74ueWQHnT1oVZPefUY9d/nyGYtAg5ytarxbo2BBsKSNmlkcGdJx0CFJk8PzobnkY6aiFRD2CdaQOQwSeRhXVH6tyr06hEvX30m52uM0aNB05RFyIeU5nM3nCzIzVxlrZBvZNHofoK8u2RG9ZWysPKuiN2Nf+hxWMJYdZue5u6LTDfoEegGbhvQRJnmOV1i+sT0uUcdA9M9FjzeQQcSdGP9yPcny4PZfsy/nI2ArnMW4hM0q9JcHfbhE2XijEntxiuTqNlSRIZn7OMDTmXMWVjpTQ8hJQW81+RXxI9QYWZB7tbEEHkVzeXxCA7XJ72OZtVzQ4blw9VDBTGx/cdiR8SJ9F59OeMUlWh+akaR/2LS7gz3DrIqc7pBDiOI025oyDqrMFr47A3tIfeTqVtTeHzQbYFYFxeIOhJRDZZ2678rWcH1BpTMmlr/vJSfgIcY8+AVqey9bqTfTgofJRMow2uhONiMzp3JCqYwjMdH3CFj+AhbWPZWh1AJ30jnVwfi2Qqa3tUKOFYU7PFEYCpBvm66GCdhdPZpJ+k3Z5Tf2OB+DxoV+LDndnavh0Gig+2MPVAd7Z28ozoiMhqhF1SV+WbnFdQi6sSwZiyc417gy/0UvqUsKfR3pt9NgGEAZrq2hgdHrfjn/wyC24U+8ZSZwpsQenjFNym07K22IWhLBlRR4PfhAtHhQjOfLj4YG2yAsB2qEGPDAAwoBzRkgikus0Us30mLbieZlVYQ1mpq74MiyVk4LLT326pbZcep06zHdPKi6eeau9NvUxlSPyWB6dh+ip+haByYORPR3X4MFIfWhWos8nm+sr93xDFsxATbkVsGOM3/7pDzE3BR1gur4YMpCnUY6WF2dQoSZpEJfzC5g2ud3Gx4vlFNTKi7OzDHj9xctX9L8xQwIhXvP4XjO9A3Y2SN7j3tmGv9cxj+/FvfJZCyyLAeokr5c+dTairSvpsHQ6rEaeNdFynZ8vp1O+T/+ANC7xr+HZUUp/qXK3P3QuosyQUrtQUta7gMnvt9H3dDKgdMgXbzJc3gQb3VS0eFqLTqykXfIAku0WNgDdTbM4y15lTC9r36uVbjoiXTQulqB2Q1m7ykAJLYsq/6c/AbhO8xU5xuWrelW74zsVtcZjahC/015XODkA+cCKwTP1HepFTh8St+/c9J7KaYN24CdoN+3iLMgi+6eKCiTWyUE/qpCj9Cj+o1BBXliDaMB/Yx2j73qEF5+n0YvpCArJWmHncluAOvchhx426qQnf7UOj8IOHIvLikkfK6HGO1GsboBatEvZr4Y546PmSb3pydib+hL63zi/H9dDZ359JQPW7dCMq9Z+5QnvYSnUtNL2HhywcvffxnOFVdCbVu8TtcFP3aIfTCmG+w9ap/BLI1MLVdDMLlRH+UOGqvW4jT27g/3PF18EQdEcz51lxmvhLL3yK/rLAjvjvfJr2DnFwvbJlnu0Tmmb8pqcdTtUCrb1WlRzM7nKf/3P37+9+Kk74MkAaczX0rDn7culboSKWCfPIQhYZl45xynmLYz/X4cSBpFfN7BRSL4pqlZMxwC2oDwh9m5EsX4qdOkNqjc4HWl8iyheG8f9vIIhS2TEc0/t6/XkCmjzCsafUfn28sVyoD8n9umUpsijPkc18OPb0qbczh65YyvtJ02/7Cd6git9OTBb1FzyERMwiG8OMVCQYUuXlIMHfQD23rl0EmZqGOGphJG2yU50d3Xzfh7tdtn39foAXMwdcRzHryv04Fod3vzwQ/TdT+++jwhIZIDQBqA+dFb/omgPO3OZ8iogCOtaSPc0YI3oK4YRCjB3eMQVXR02G3ReEGKt4iuKXfQWKuFY74pmDUBbaea6uxGyLn4uye9HOeijXosYyqK/YIgIRUDU+HYmHW0aOYgKfUCVkollWjn5M8VvzPCkp1f0I47vpug0BuxgsS3oTlkJvaBXFEHQYadg+OjETz9mVXEPra+bek/YqiN0O02xLqi0+ijSjkJ5ad8UtyVhYg2KqthjCdHcezhWcTaZnjOJcTW7nGAVDvJmh9q/LiARlKuPsKCuYUpgG5lBMb9apgewACX5rPfVJcFV3ZY7YLwYMAPElbSd2EurJ4y/Q89seKFMoxHM8PawZ6+IFkkNmNvlJPagnjtFfWmLh6fUEhn3YF+HL3iNNEL191zMlLoAm0YaMPqWUr2Z04AGwXrslUi1EVH1ABDzEho/y758GT2DP/j/5Dw7g3cY3wL6OvAh/LEv4Qv6P+guAKTsDM93Jf7U2lXzg/Z79P/kk8uWLuyjkIjmQ+YheRrFX80nzDpAh9WaIvaNUK8TAnoJmsgS9C1gDMlUNXQp9Q11yJCnkepfn7DI0VE0iWnJQNDax9L1m5KAFBIkeZF0Hxz6RJ9vQLW5M0b5SasRoW9cVfG/D6AMF/Q1mqE5zvTLuklUT1LTriI0VFAQU7ncLYfXX2Khp07n3JMWrlg9oyfpz0mmDalmKQQaBXdTYtBdd59XdasONmGikVyk+hvCjtKtwh+R+ckD0CFM/TZ6qz3nrkUNzAp4V2nC3vCUFw1b1xgRhweFuNjRF6itidsBz4XC7RbKKWjY8wjqrQ6VhKprvbhAzrizkgGIrqpbdRZN0uOAjqqvv/8x80bubOnNXoDp+g6S5MYkURsXiZ6jEAwW3dqNwFHcCrJcme78t4KKzHTXEt3PMGhE7YUkU3Ud4RWcbF0W1xg3lqzL7flidpECM9heLGC/n7WHLe770e9K8e0pFL9NLoDhwMoutnv4jPYv5ECa35J0Q+1FNhr3mC1ytZnqUq8SzDioXOWKVftt9Msvt0Ikb6HW2//7bvrLL2h4g8H8AhASmLeumP6SRUVX7C6i4rZGSzC8ZfV3omhmKDqBMTTlrcTfqq6qYt9i0GV0PkP+KkHJUE7ZVxnL2YLEZNBg6waU1ET/OhTkEyRDKK/uvcDQ6Bo99FN5JgRI3rXWlNS+Jw8UvplzjTB9M6SaLmVwApQvAR3uW9wrLdMjVc/IXDXrAzw7WvUcC/Wq4vR7Vb3NGtATkFMa2OwqqpTiTmKjQt30OrsVK3jO0SiYILZSBaW3UZa1aPITBJcqypqyUIW/oQu09UVvD/s9nbMbNkc8Yh49IBni8bSSIkz9rjE0NLEa/Oh2YFTkjEtcWUaplvmYDLoT5fVN55xlE6NQIAxTwa7nPa5Lh6kO0KEeqTMistmrcNOa9gN8e2XleFj/UMhQQrxv3VKWlp7cNkZ7Vb/nnIAi03mptISefWAZLGbtD0ufPvUcjHXNDEt1JVsDB1ndoL7jT2EaaJIF+TB58bNUKt3qsOzkJk6Jha0odqRl8pdtt+YCmAAN4uNpEFV1h6roMG5YT/CRmDqznQYolOEDva92IA6bQ3fzia1qDDD7z/JI63K5m9hhaT+q34tdPI++ybatkA3ySTMmyp8Z94vRSUnLQY+ra5Gq/80Ast78lNsSJAwMrC8IBukqQPGKZw6cSk0znPKEd1eeTKDr8mCfB/oZRoXuQRrBlmcB26dXo80TS9UGuhDFDVTBU5WhKkpILQeqXhuL4DDFDVdVDQ9XDbb+qKKWYW+IJ6ugZCleTn6bSxCFMtwHzRn4JrVBRZIwtR+265CI8FJVQseP1LciJ/pITpE9wORuy5XZ0cinyXH/gZ7N+r24n6tYpa5OJJw02gFSrqp69R4D2Pr2PajERkpd1QOdcOustrPkQHObChH3q8jltTwhnIdc0FSJAHZ+BVns1zFmtK8/iGaF+gu6UoIaUkV//Oub165py5jSDntUO2HrIkpM7qHYuA3QCZm0U8fEiLbIWMcah42ggxVONxxOLSEC8fsEiv8eGPUcdu2/DkL8LJKzaYBI1JxdwgJTlPKoiZhxc4kK1P+l3kKjq0G33AlGhkzBKQ5dvQLysX2SMHM6fJe/M/xtWYI8l1fnIWpLav2zCTtrXlPukA7rQof6en6u5AyiFAZ59rNUm3XmGMPOBGmMiV/DHt6m0QvrpjC1pzIue1cuI2hTOrv4PKCwIwFLQ8SAcuxYnNPI1fuscSPVS2ZQELvCONNEn2j1QJuSF9Fl4IADv+t9cy967MRjGe71NXB0QFtFt1UeS00Qltwx2/R7NJIw/pMxJWsWhya39WElMH7AJFAy0GKLFdoymb1lYDelcl0AZkz1qRPrKAuU7QY1VZEg+qcY7xL4ZhqkArb5rxbc2/CYt+AmNN47THpT72aypTlRweIB//+Ykg/P4sG0xwIjzEgwBGl4Zvrzb8yYjAKm47P0DctUdTARztU95lISJQbg2Bb8qQIKOlS4X3qIcUyxStZC2Da7iCmG3MJc35aWkdKATXE7AY9PYHW9Q6mA2V/JdNlPnXiE8i6Ifa3Z9SeK3IrSDc1Z6qFBH29Zod4D+sufsY78TC+yv+jXaveszg68UlVjThWa7Ltie7UuvnuraxSseLHdZ0iRP9HbY7LfHv19mi4gEXIt8+MoP+8R/WBYRFOHPk4skw7Zcqok97Znzyg+ip6MVoqLRo31URJfC4irSBqdacw00t6Niybbi2YDG4sDbjFY4hI8DqFwPKkHuC7blUpIRaAW52xBBjQIA0PrAFO3NOxXftZekcPydX5hBayht6NL6qgGcZIWcVSTOFGb8DQKQnVVdDbO5FRFYTK29TRqB4NtpW0f5VKrcGFKhQI9ZwGfUp4f0SysdtF7HbQSmelPj++leyW0ltK3jPaWcTrksU5cJKM/Upj6ygwrBfou/sgTQ3nTvnbjKj3Op1P0mhF9xgBdeqmxgHHLjHHkP2WkTm5FbMtOAfiwYXWHBqPHi7sW4eBBc2iqyT4o5vLZgi1m1ktkXJeSay2xUEicws46UDewzbb0GA55Vs3J7bppTCfnGG1Nn0j/D8vscfWeRYoV4gPGKEjO53uwNBj51iNGzIaYrQ/bfWCp0I4q+JZ2mcqlRAqNdKyc2FMxYNbjpVBqsAGMlFYixpu753qqR2puy92hI6evJCBaopkWPWjPfIUiLAjqsfc20NtNdWhviNGHFri2fEjxKOnHDCBEWXKwZreKyfmUTr3biIac+3HFAuGjwqVOk0RueeKvo3iNhMGFA9z6sVrH9CIMKNLhdCZoDV9KYcBe6syYdrWwj90NCpS6Wgc1Kk9BwlHMT1ZxdZpTh51S+IrDG1RMDGoXOsGFp6Q4yUr4Sk/cSN2uSTgUkJUyJVycx+gPgo7VJlq3v3fU3R3flPxQ8zNIGaZDyVqUv9CaYnspEZeS/vbk1swYRoupjLa7RLeLYd9sUq1cclN6mNwdznJRiTwsLJvLw4D0U3gE7A98yvTvS8qCoToxXUoRqB4pEZNN7mFCln1j+XhADNboH8kdqyGN4KcUZMcip3fkeGngihKzqpi7TrwAbKJyUhh6ZGaoXiaUcshdxYVNTws7ceNMenVVykwnjsoNnOpVeWLSnl798dAWJ5ZF1cVVaqqbeEXUhctd4oeWKmanV5d6wtUFcAUtCesqTWvC1Y5P+I6IGfhk47/C3/uhABo/4+XlGVq4rMP/LHpci1AwtFz2tsVkgUhp+Bfdvy0MJ/BWx8TpOTOxcZLlskQM/cjt1JA0i5G7ZO2HAmn6B288ekl8wJXW9dhEyCraD8j3zqYxvoz3xqvBRbH7qS9J08AGNP23hJkME5MzFh2u0d+8mfiNofNKP3qMBWR9ZBAZIPpI/NivFcmlsWNzSVv0/EfHbTkDdxnHf0akFg9VMdEqXiTRx6PNw9ivtJJMnw1z7QVDgRDeklD0IpGUHg1br6C1ZSz8iHd+6hzt2VPiJ8cnOUBdcuEtuF8Gm3tSaFOYOCUZfPHFk5oMhUCN11CuB08g8RPj0zDf+NlyLCeaN9RXp44UczFCm7eiMvhFs2rrSHmAp823ix6Yx568qg/d/uA40x6UoRvvDklC0rFd+MJukBajZ8/kOhjRBz6qB76a8FE98JznXJRo1pExuKv9IZkGvaRItefjOV5dN6cz0enhYz35Mi+u8eIVHvHo5WNw/MEk+Y1g+hNa8vzQhlpaN+Wm81ugl8mwJxvTSfWGKxjm37fCmbTgsB6MRtovJXcNUMYukkAhpfoZzy1pkwwavk5yNBvWe3uBliE/M9/N6xRbW8zRqmZ2La7NYBySU3BDYPhcBcA49DQCRs4+xlVIa4oDZDKGKw4Ztkehfp+ED0t/Ttv0JtzxxwBVhwwemkL9926tHptStfz3XlvS8GGaoMdwd0jQeV1x15Rv8vC7YEsrM5m1ehyzk6F7gdjuu/uexUxlI9IX+KCX02R0l4TuFCE0TycDB2dutTA+Pd3RtCHRO52EZECwNxZHHs8P98IW/0QeP8jfp5nUqdxOfWQrg7zdbeXjeLsDwuWxAGuUkT6RcbpN6aK4yE2a1yD/c1DoFA/yOWW1N0yNSX6qPHOa9kOOeEUV4shKU4TkFwzdbn8ctiUdltT9HMzNurii3bj0tOfcF7gfezIVvlqwIwKML/yQB/n2kntlqwGF2tDfxhoYrs9b0Wh1hBrhAIdi0DPWkK5qRcCSuxD3/G7jYrUSe0yQDKO2TA1Vh6pKCNuZvPbAERwxfUFHZgpZYj7vxAzVgZ5hrLym0U0ob57k9Pyzs2JovEaKud/64syvvD97aerCFhe6g+YXTNPpFkyjl0EwzbYd0ItUHNa/mi5h3v3H1y/LRMIbHNRinLUyOa6u8IXJyg/oJezNE2n9JKLVrRjCPa3W/suXju7izhyVSaMv3RkzlAa1zO/UcQK3xJ23tM/CONFyFbgSYewA0Q8XbxCie2GQgnsZxro1YD1/MhsykEfYyQj4k5iQaePIvI41NMiEnCNwObsoESQSp/6laXbGhuZq8DD3CXzODHigxDLA9Uwd+WIZYoCmjHqzDPFAU0i9WR7lhLZG4PPyFJ5iOxYusTyF0fTIZATIAPc5sk5O5EinrImncCsNbxLcHw5Us15QgzxHUhfe3IPxFB3d36O+BakankxCClt0mEp7QdNxcVeUGN+Rm+tk8xfr/V2Ot67GfJUA6v4pL0lwYerlaC6zzdUNRZLXqZSLQZeQE71NyDGYuW74nrHDOFLNqPt8+l8HOYO9LQu/qnSQU/5Ju5hNwmYYQws0jTlCQbKRSXrMq8Q3ZdJQoSD9TSenulSZ+cY4QkzpFPxsySqen0h0fv/2N0UrXris0T6EC2v0y2VDP33Lq5e4mxCs6OoONj8ivynxkuJ7yuSp095jzrawB4KXOUht4Km8R/82Qoddo3YXY4DJXQUMcxHH/HK0Q7eZfRG+Y436ibvMVXubvYH+/J1emBvWNqWo1uQws8AuJ5RY8GzJVD0JIaM/GALlXKvFP1I2O5lJVuIIPWNgaar0kwloy4AYfctx9gM2ip4zg9Qu/e9Ngm95VbC8IA5hyUw6ubwQZqqX3Eh5/MpLb2UKd+lXbrPRhy4BntH1k7xDUxfGrQxu9CCYa3llfds9G4LidqJuOMSnxJTofPTqQqv34l4n/Fw8OG08pmxp2G/wrh9XwjOk21TOdDul/aYmA9Qo4B7Aou2dV841WHRZ1uDFltYAhsDcipjcqd8EW6EmzXvoAjE+ad6NYVMeCQnVQ9n4eQpoC8fLoB/Kl5+6QzKZrr33PEOjhwRKn+69NHnL/cJCrLmaKn0yUVCYmMGJc6kSHWl5xgiTdjHpdc1xk2cqyYeB0v/FC6lTzUC5c6ecPY7spU90zg1DaRJDKRJDDaJLPq/lpEkMVODtSKkA2j1dmWwQSAGc8gJA6RNNn0IBnHJWrDus4iD2Jikvpvgh1vOFAhahpig1sX00cvCOYKJfdJ6aPvajjM3Vk97sO/HG2ocpCBRvLaRXDA+Seam+H7kqkN2IafLCKRKVLWE2EgNMX5khS+QD+cBH15zK0X62tPnBJ4PeBwNZ0/mK99KmG+g6xbtzR9KUr0Rk+naCMfx9Yp1YfH/Z031l9QRzH10rX5S74sSJEXR6cixk0fOsVVKVO9WGrkh07zrxWYs+xceR2hd8OQZEgyrvMm53zyr5Pc6Zssz10+06Uyk9l4NAHBOgukEjVI54sCqmWCwvx9RYCUQhfk1eD1ITUFPh2O7Kpu2MLWiMhmO1NgzLRuVJZZR3Lu5lS2jZry4Xn89LHNb9QV3sl6uLq+Q+GbVmNIfT8a8qIA80zNUkKgmkxeCjyTlabWgjl/gasbp2zhzViHuRvPBdjmTM1ouB7GJhzxAN2cbhUQou6GGiE3elLBsXi0MZz8Eim9fHMClBZYd5bSuajsfmqGvz6Nqx/+IH6bBermTsoExsphIJ9ouo3FQUh3OWnfcLjGeWHRiHqe0NRZmpQyFcanC23WA4uLkQfvCiTTnTHXlzJ+SNo2eZEnwpTVm1dsmuxwxcA79EvnZ5QQSiEgfIGKL4J6C5GdIcpZHAm4UkjVI9lEbo/r8uNxSV0pXkgCsziOLRIcXq6G0Nadg5agqSfvs7mrm90hS3XqbA6+b6sAXwP9KXhPNxIFtyrcRboRa9Cm/EpjhUXftnUe2/0YV5OiUCiNfLY7+oShLPZuivNgORAbRNkQZqgyrpY83COQfqk8I9A4V7Roz4Y6BYTjKbaS6moFlZbaCaVzcw0IUrl2MMe8bMfcA/MAyvE/ISeSQspVGYi9cDd67/zrtRL14rnGKe2d5Qw6J8CE0IfyY3Xh+LahAInwhBGoNm1uD1sYCQk85wp/JJAD69HyhiZ1IN1yAoC66auIVzdXAAH7R9CtY8fzVaE5j0TMr3UOWz0brkij5DV/Rg5Yuj6xWvTg/V/Hy0JhpmZmSdaz+iXcy2PSP73RiME3pQGVajbtzVdV+K2cvR6hsQwd1hJ47AuTgZjs7+PQprnA4oBdpMxkQPwDjLLsb7A4ryTCoYnwhI5yr7daDREkXRNg7n4mV2dgKvAI45Duf87BQ4190JXToV0vFOjULickvrTDNSA9hFhzc1ajCLsArJkiWpBk0xV8zFOv/vjJQHqXFSklXUcGQyixnpIu0BtK3bskUD9wlyCWWK2s88nZFhZXWuF2S+Z+PVlRXmGD989flRTkwkwU7E8PcQmR2hDQSnT9p6cAep5Pw4UGV7mtGh2kgHjy2nqr6eUYx8EFkvxyujCTDM/M8uXp19efbliVpaV+9nmCN6Zk5vmK2Rch4uYjz2wE36oU/NP0H9qNjgqQNmCKYtLbckWMVA3hfQwut2Uwqdtx9PpU+hbnW6MKMznDTUsyPYUhuCsbrm6IdAcMVfZxykXP7uLhYL6CvudWltXJOmWrX9dW8yVTvi0EmQ1pnoll7XX6N3EOLZhe2Zh28wJnUxt7MRslePf6VrUZ9HcVVePacjxPY5vs/29878qRMGjL8N2yi84WnHVjz5MjcTL73zkuO3O9vLnRX4iMDPo9i5SVvBU9tMFciw3ZZ0H+ThSkUfZoQJFSxgsX8ZX5dIQHEjbqX6jg9//vo13VW2ulsvPNt/hGlzSfVVTpCope4tgfD2f2ONGsj98z/+5fvvv333lDOhr7XNlaSHAvsQAPqYwoQA5tRHWbJ3HGSzlS1CZ3tTtt9mKToeYlMP/Zv070d5Z+cOddOUpZ1w14puhs7JQ0vDSeKGKYGQZEzgQHIkpdlrmWLyTz/+FQ0XllCAOG0Kigh1yhkolUAm2jMaea3MZtJh8HhizmCmLE2EMWPJx0SmLFKEBqq+ut5ckoh6zig881abfszr7Xs0/QKJYZYWmZ8jEh/Ktsvr9wxv/+4j0/8/TgRlgf/Ic0GZpiavyZbSc0lzjwNtWHU8l9BCgdaxuo4ZyrhpY+LdYZvrLe3ckjkmT2L+VYC8rdjWoPP0AIC22wKJo79KGA6mFkjdIyn/M7fk8qFfElsSOJ5NgekiYxz9BUMSPNpMfYlDExin5oKbMq8iJjZtZkrjeTQigYlaaUmQOrLaH2KTOTSvd9X9grxo/FyvqW7XHK15Fy8rp59kqFmePk6FufogAhF+LvlaxpWGFYh+otqTklI+7ba4QNqbJ1xJPTSiqZeax10mg86vc7s0Q9+9U5YxP0EDZ7CQB6zvxMqA9D7yJaTILZwMXjJdfMhluaHc71TQvumX7qdel1zPfd2vx3Om2+kibITLqiNENrWYySBc1mZHt8XVu+EavQbUO6fGo+Px5yIXc8Bti/wWmR3N9jmfTNHd1OteZvqY57jUZgFAG3W12K21FwToDm1XVpXyiXTyg2wGR0kZW7SMDYwJv7s5GjG7i9OnTonMIx1xzn59xoSnkRcvX7Gwb/lCbgnCzCwAUHNFPEh23/izKfVe1h5TkXtln9JNr0bvRFnpWCdAchQ498D5dCBMq3McaFqpaOdofOq7bMZMJTHrNaSlxMr0Pg/rK5922O7IYV6aMlMq7oWD91NZxk4OS8Oj+EvetmH2eBBvHhxXBUKSp2U9Ttytkkk9C2t+KJN/KIG4p27oOwVMGvF+553bVXp7sH7bfDNmejm8K+OHqV+rKxhB00ATz2G3uil215gAGl1rZzAEbuWpdzL/mrUBZVkW68T4rahkcOFg8r9AJqFeLg67ixlXuQeSCfmJgswq8dV480GleHU/jOUj8mmIz9QkkNnyIda4cd2aA/EluuDUmb3JUEpHhXlc7cwstzBzYT9TqJGdGht9pIvycCMGlgw7C7vHfW7zet9sm4scC2V74CJcaaaTJ7IsaMd2xjHJEU25oafR2ZQ1putxMGwI8nZZ5yIH3ft4wqLhilum2g5FDRh73bDxbCDvFe+d548e6pl/e5NCt9tKz2udTbqZBz7nyl39WH48ebGuY2ZMehdhKBaEv+WoOOtOBu7yVZXsG7eqWhPIRMy06tza9Z5vb+w+SObUxtUsoxPopls5gjn3a1QfT0jMNnB3CMszLo3duH04IUexa5N74m3WvsZ2eiLjoZuVafJcDBgfncuHmBpAmeeNLwUB22hxR9+q5tG2hgrg3KNZef46hqNhPJ1yCfYgnk7ElYsvFxE9eCdiRh8/5xpF/SwX4ezEFnwAceF8vqFmTeWq6VfyOhPI+27MFfIKgdfrYvt36W5lrjDBO1iLezw8ZenK+MW/C3IMZCmRo2dq8aU8oSG/HVhlk+RplHv3CT9/Hp2jZ9g0kDzb63bw5gOXgdfuFQrGRESFVY5+an3B+qAvYF649zD7VzgH8q4bcBrE0P2RfsZwMzL3kgZlSk5Ncvz+9VK5vIkT1LJE8Vgvtzbjsp8tonMvf9Cxqwwc66fkyb3rMj4lhb26CaH3fmDiHHIIfSqC7329j40/lB58MD3+2FbCKWOSoKcjCcccrWxULf5Y7IbU5TG1+YRcnGPoHFSnj6rVJ6vXQ2p2GMODii0LGuypttaF9q6XJ9CJknQU2s/4SgsmbxrNSP/s2cNG2QgwzuQxnrPb8tzgEr0U7d1g/dZumUXGDc3mYbpHorMdaIOR2gxgWHwN1OwVHmp5MLz7aMsDNU9uecDae7TdYL2TWx0zEB9terjyE0fNDMon4/lY4PmxllkY8SnR5KFVJiqxsusc1xv+DZT8tLsW3DsW1H5L5/ACxjEN3G/BNv1QYvRYnCv3xMK+Yhv1ef8eGL6Jp7/hIiGBw5MOOYVD+22Xyw30Q+3E6W+fbwd24c4u7ciO/Am78xMEZ2jXfoyTH9Ed+sJ9pFe9nf2RzHmBHf+w8HND1weNNJp4V+0tKJrqqUeGp6xHiq1QpmDYxqNvmVnS84COQTbG11XVu11exW78jswBe2l+LFu9g5cFRdFU99q86M6La0IwTjgOEkG/eu/uoVktb4dLZZWDFt6w/oRjncGELt56PCmxy1rsq/oeHdvy4YQLT0224HWEW9rDhsm+kSr2YPSH55fts2nLwvyyWtuyDz1oSrWyD34JFqQwtwsoo0tsBouOH7AYKFP3FC+ca+KRkc9Rs6mNdKTyGQqO2AcgczHkdA09lyxUggkXdIL7P7tQGokjhuljSV4HSVt24UliO0zZGlDoawhKgPLdvrBPofp9Mg4Z4QfXy3E60r0ZKOD1ydPr9YwGzcBh27+c3A0GxUn/wKB77jx6MMT8aK/NOZlDMc8NydLfmPmCEaPnbBvR/djQAfTKq9Rhksr3Iw+NKGTv9fM/SNbfasbu2hsV/L/ZUeikQEpAmDH9LlrXMuvKYUfNOY1kFOMHY8xzZAF5TsbTPMe+5rmymkqX38n/AxxRcuMBwAAA'),
}
for name, (expected, payload) in embedded.items():
    contents = gzip.decompress(base64.b64decode(payload))
    actual = hashlib.sha256(contents).hexdigest()
    if actual != expected:
        raise RuntimeError(f'Embedded script checksum mismatch for {name}')
    (SCRATCH_DIR / name).write_bytes(contents)
print({name: digest for name, (digest, _) in embedded.items()})


In [ ]:
# Fetch pinned WHAM/YOLO assets and normalize the licensed SMPL filenames.
import shutil, subprocess, urllib.request

WHAM_REPO = SCRATCH_DIR / 'WHAM'
if not (WHAM_REPO / 'lib/models/wham.py').is_file():
    subprocess.run([
        'git', 'clone', '--filter=blob:none', '--no-checkout',
        'https://github.com/yohanshin/WHAM.git', str(WHAM_REPO),
    ], check=True)
    subprocess.run(['git', 'checkout', '--detach', '2b54f7797391c94876848b905ed875b154c4a295'], cwd=WHAM_REPO, check=True)
actual_commit = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=WHAM_REPO, text=True
).strip()
if actual_commit != '2b54f7797391c94876848b905ed875b154c4a295':
    raise RuntimeError(f'Wrong WHAM commit: {actual_commit}')

downloads = {
    'wham_vit_bedlam_w_3dpw.pth.tar': (
        'https://huggingface.co/camenduru/WHAM/resolve/main/'
        'wham_vit_bedlam_w_3dpw.pth.tar?download=true',
        '2ba0cb6a7dd597023a6b2ad6056e7a8b6b33144a35fabea570bfd00842cd4eaf',
    ),
    'yolo26n-pose.pt': (
        'https://github.com/ultralytics/assets/releases/download/v8.4.0/'
        'yolo26n-pose.pt',
        'eb3bb8268828aeaf515cec23a4bfafd793944a86fe9af94ba7823609c14522a9',
    ),
    'J_regressor_h36m.npy': (
        'https://huggingface.co/camenduru/WHAM/resolve/main/'
        'J_regressor_h36m.npy?download=true',
        'c655cd7013d7829eb9acbebf0e43f952a3fa0305a53c35880e39192bfb6444a0',
    ),
}
downloaded = {}
for name, (url, expected) in downloads.items():
    path = SCRATCH_DIR / name
    if not path.is_file():
        print(f'Downloading {name}...', flush=True)
        urllib.request.urlretrieve(url, path)
    actual = sha256_file(path)
    if actual != expected:
        raise RuntimeError(f'{name} checksum mismatch: {actual}')
    downloaded[name] = path
WHAM_CHECKPOINT = downloaded['wham_vit_bedlam_w_3dpw.pth.tar']
YOLO26_WEIGHTS = downloaded['yolo26n-pose.pt']
H36M_REGRESSOR = downloaded['J_regressor_h36m.npy']

SMPL_MODEL_DIR = SCRATCH_DIR / 'smpl'
SMPL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_aliases = {
    'SMPL_NEUTRAL.pkl': [
        'SMPL_NEUTRAL.pkl', 'basicModel_neutral_lbs_10_207_0_v1.0.0.pkl'
    ],
    'SMPL_MALE.pkl': [
        'SMPL_MALE.pkl', 'basicmodel_m_lbs_10_207_0_v1.0.0.pkl',
        'basicModel_m_lbs_10_207_0_v1.0.0.pkl'
    ],
    'SMPL_FEMALE.pkl': [
        'SMPL_FEMALE.pkl', 'basicModel_f_lbs_10_207_0_v1.0.0.pkl'
    ],
}
def licensed_asset(names):
    matches = sorted({path for name in names for path in KAGGLE_INPUT.rglob(name)})
    if not matches:
        raise FileNotFoundError(
            f'Missing licensed SMPL asset {names}; attach your private asset dataset.'
        )
    return matches[0]
for output_name, aliases in model_aliases.items():
    source = licensed_asset(aliases)
    shutil.copy2(source, SMPL_MODEL_DIR / output_name)
    print(f'{output_name}: {source}')
print({'wham_commit': actual_commit, **{name: sha256_file(path) for name, path in downloaded.items()}})


In [ ]:
# Fast preflight: code math, exact files, parsed fields, and raw-image mapping.
import os, sys

TRAINER = SCRATCH_DIR / 'train_deployment_tiny_pipeline.py'
common = [
    '--output-dir', str(OUTPUT_DIR),
    '--cache-dir', str(CACHE_DIR),
    '--three-dpw-root', str(THREEDPW_ROOT),
    '--sequence-root', str(THREEDPW_ROOT),
    '--train-parsed', str(TRAIN_PARSED),
    '--val-parsed', str(VAL_PARSED),
    '--source-checkpoint', str(SOURCE_CHECKPOINT),
    '--wham-repo', str(WHAM_REPO),
    '--wham-checkpoint', str(WHAM_CHECKPOINT),
    '--yolo26-weights', str(YOLO26_WEIGHTS),
    '--val-tracks', str(VAL_TRACKS),
    '--val-frames', str(VAL_FRAMES),
]
environment = os.environ.copy()
environment['PYTHONPATH'] = str(SCRATCH_DIR) + os.pathsep + environment.get('PYTHONPATH', '')
subprocess.run([sys.executable, '-u', str(TRAINER), '--self-test', *common], check=True, env=environment)
subprocess.run([sys.executable, '-u', str(TRAINER), '--inspect-data', *common], check=True, env=environment)


In [ ]:
# Deployment-aware training and validation selection. This is the long cell.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
training_command = [
    sys.executable, '-u', str(TRAINER), *common,
    '--clip-length', str(CLIP_LENGTH),
    '--stride', str(STRIDE),
    '--max-clips', str(MAX_CLIPS),
    '--batch-size', str(TRAIN_BATCH_SIZE),
    '--workers', str(WORKERS),
    '--yolo-batch-size', str(YOLO_BATCH_SIZE),
    '--feature-batch-size', str(FEATURE_BATCH_SIZE),
    '--initializer-epochs', str(INITIALIZER_EPOCHS),
    '--joint-epochs', str(JOINT_EPOCHS),
    '--last-stage-epochs', str(LAST_STAGE_EPOCHS),
    '--log-every', '25',
]
print('Starting one deployment-aware training run...', flush=True)
subprocess.run(training_command, check=True, env=environment)
DEPLOYMENT_CHECKPOINT = OUTPUT_DIR / 'tiny_pipeline_best.pth'
TRAINING_REPORT = OUTPUT_DIR / 'deployment_training_report.json'
TRAINING_HISTORY = OUTPUT_DIR / 'deployment_training_history.csv'
for path in (DEPLOYMENT_CHECKPOINT, TRAINING_REPORT, TRAINING_HISTORY):
    if not path.is_file():
        raise FileNotFoundError(f'Training did not produce {path}')


In [ ]:
# Locked test: run the selected checkpoint immediately, once, on the saved population.
EVALUATOR = SCRATCH_DIR / 'evaluate_deployment_tiny_pipeline_3dpw.py'
TEST_REPORT = OUTPUT_DIR / 'tiny_pipeline_final_3dpw.json'
TEST_CSV = OUTPUT_DIR / 'tiny_pipeline_final_3dpw.csv'
test_command = [
    sys.executable, '-u', str(EVALUATOR),
    '--parsed-3dpw', str(TEST_PARSED),
    '--three-dpw-root', str(THREEDPW_ROOT),
    '--deployment-checkpoint', str(DEPLOYMENT_CHECKPOINT),
    '--wham-repo', str(WHAM_REPO),
    '--wham-checkpoint', str(WHAM_CHECKPOINT),
    '--yolo26-weights', str(YOLO26_WEIGHTS),
    '--smpl-model-directory', str(SMPL_MODEL_DIR),
    '--h36m-joint-regressor', str(H36M_REGRESSOR),
    '--pose-batch-size', str(YOLO_BATCH_SIZE),
    '--student-batch-size', str(FEATURE_BATCH_SIZE),
    '--smpl-batch-size', str(SMPL_BATCH_SIZE),
    '--output', str(TEST_REPORT),
    '--per-sequence-output', str(TEST_CSV),
]
print('Training is finished. Starting the one locked 3DPW test now...', flush=True)
subprocess.run(test_command, check=True, env=environment)


In [ ]:
# Final original-vs-tiny table, validation provenance, and useful artifact bundle.
import json, pandas as pd, shutil
from IPython.display import display

training = json.loads(TRAINING_REPORT.read_text())
test = json.loads(TEST_REPORT.read_text())
comparison = test['comparison']
original = comparison['reference_metrics']
tiny = comparison['tiny_metrics']
display(pd.DataFrame([
    {
        'pipeline': 'Original released WHAM (saved)',
        'PA-MPJPE (mm)': original['pa_mpjpe_mm'],
        'MPJPE (mm)': original['mpjpe_mm'],
        'PVE (mm)': original['pve_mm'],
        'Accel': original['accel_official_30fps'],
    },
    {
        'pipeline': 'Tiny accurate: YOLO26 + FastViT + split WHAM',
        'PA-MPJPE (mm)': tiny['pa_mpjpe_mm']['mean'],
        'MPJPE (mm)': tiny['mpjpe_mm']['mean'],
        'PVE (mm)': tiny['pve_mm']['mean'],
        'Accel': tiny['accel_official_30fps']['mean'],
    },
]).round(3))
print('Validation-selected checkpoint:', json.dumps({
    'epoch': training['best_epoch'],
    'stage': training['best_stage'],
    'checkpoint_sha256': training['best_checkpoint_sha256'],
    'validation': {
        key: value for key, value in training['best_validation'].items()
        if key != 'per_track'
    },
}, indent=2))
print('Tiny minus original:', json.dumps(comparison['tiny_minus_original'], indent=2))
print('Relative change:', json.dumps({
    key: f'{100 * value:+.1f}%'
    for key, value in comparison['tiny_relative_change_vs_original'].items()
}, indent=2))
print('Detection:', json.dumps(test['detection'], indent=2))

bundle = Path(shutil.make_archive(
    '/kaggle/working/deployment_tiny_pipeline_results',
    'zip', root_dir=OUTPUT_DIR,
))
print(f'Download this bundle: {bundle}')
print('Inside:', sorted(path.name for path in OUTPUT_DIR.iterdir()))
shutil.rmtree(SCRATCH_DIR, ignore_errors=True)


## Outputs

The ZIP contains the validation-selected deployable checkpoint, compact training history/report, and the final per-sequence/aggregate 3DPW test metrics. The test is intentionally run only after checkpoint selection. Do not change training settings in response to the test result; that would turn the test set into a validation set.
